<a href="https://colab.research.google.com/github/Keithmushininga/Keithmushininga/blob/main/Trackrad2025_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

tanyamushininga_trackrad101_path = kagglehub.dataset_download('tanyamushininga/trackrad101')
tanyamushininga_clean_trackrad_path = kagglehub.dataset_download('tanyamushininga/clean-trackrad')
tanyamushininga_4dxcat_path = kagglehub.dataset_download('tanyamushininga/4dxcat')

print('Data source import complete.')


# TrackRAD Two-Branch Cohort Analysis — Corrected Paper-Ready Notebook

This notebook is rebuilt to avoid the previous errors.

It has two branches:

1. **Branch 1: optical-flow dense POD** using GPU Horn–Schunck apparent in-plane fields, classical dense POD, enriched dense Triplet-POD, cohort-level spectral/rank/event analysis.
2. **Branch 2: centroid trajectory Bayesian analysis** using manual-mask centroid trajectories, Bayesian displacement/triplet models, residual-corrected calibration, and reliability/risk–coverage.

Key fixes:
- Handles the actual TrackRAD structure: `images/<case_id>_frames.mha` and `targets/<case_id>_labels.mha`.
- Uses corrected orientation `(H,W,T) -> (T,H,W)`.
- Dense POD mode plots use the **cropped H&S field shape**, not the full image shape.
- Dense event analysis includes both **flow-acceleration events** and independent **GT-centroid-acceleration events**.


In [ ]:
# ============================================================
# 0. Imports and configuration
# ============================================================
from pathlib import Path
import time, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    import SimpleITK as sitk
except Exception:
    !pip -q install SimpleITK
    import SimpleITK as sitk

import torch
import torch.nn.functional as F
from scipy.ndimage import binary_fill_holes
from scipy.stats import chi2
from sklearn.metrics import roc_auc_score, average_precision_score

TRACKRAD_ROOT = Path("/kaggle/input/datasets/tanyamushininga/trackrad101/TrackRad2025")
OUTDIR = Path("/kaggle/working/trackrad_two")
FIGDIR = OUTDIR / "figures"
TABDIR = OUTDIR / "tables"
CACHEDIR = OUTDIR / "cached_dense_hs_fields"
for d in [OUTDIR, FIGDIR, TABDIR, CACHEDIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_name(0))

TRAIN_FRAC = 0.50
CROP_MARGIN = 60
EVENT_Z_THRESHOLD = 3.0
KAPPA_ACC = 0.25
HS_ALPHA = 15.0
HS_N_ITER = 300
K_LIST = [1,2,3,4,5,8,10,12,15,20]
MAX_DENSE_CASES = 8
RUN_DENSE_HS = True
DISPLAY_PLOTS = True
np.random.seed(123)


In [ ]:
# ============================================================
# 1. Robust TrackRAD indexing and corrected loader
# ============================================================
def find_trackrad_cases(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Root not found: {root}")
    rows=[]
    frame_files = sorted(root.rglob("*_frames.mha"))
    print("*_frames.mha files found:", len(frame_files))
    for frame_path in frame_files:
        case_dir = frame_path.parent.parent
        case_id = case_dir.name
        split = case_dir.parent.name
        target_dir = case_dir / "targets"
        label_path = target_dir / f"{case_id}_labels.mha"
        labels2_path = target_dir / f"{case_id}_labels2.mha"
        first_label_path = target_dir / f"{case_id}_first_label.mha"
        if not label_path.exists():
            candidates = sorted(target_dir.glob("*labels.mha"))
            if not candidates:
                continue
            label_path = candidates[0]
        rows.append({
            "case_id": case_id,
            "split": split,
            "case_dir": str(case_dir),
            "frame_path": str(frame_path),
            "label_path": str(label_path),
            "labels2_path": str(labels2_path) if labels2_path.exists() else "",
            "first_label_path": str(first_label_path) if first_label_path.exists() else "",
            "has_labels2": labels2_path.exists()
        })
    df = pd.DataFrame(rows)
    if len(df)==0:
        print("No labelled cases found. Example MHA files:")
        for p in sorted(root.rglob("*.mha"))[:50]: print(p)
        raise RuntimeError("No labelled TrackRAD cases found.")
    return df.sort_values(["split","case_id"]).reset_index(drop=True)

def read_mha(path):
    return np.asarray(sitk.GetArrayFromImage(sitk.ReadImage(str(path))))

def orient_time_last_to_first(arr):
    arr=np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got {arr.shape}")
    return np.moveaxis(arr, 2, 0)

def load_trackrad_correct(row):
    frames_raw = read_mha(row["frame_path"])
    labels_raw = read_mha(row["label_path"]) > 0
    frames = orient_time_last_to_first(frames_raw).astype(np.float32)
    labels = orient_time_last_to_first(labels_raw).astype(bool)
    if frames.shape != labels.shape:
        raise ValueError(f"Shape mismatch: {frames.shape} vs {labels.shape}")
    return frames, labels

def normalize_image(img, p1=1, p2=99):
    img = np.asarray(img, dtype=np.float32)
    lo, hi = np.percentile(img, [p1,p2])
    if hi > lo: return np.clip((img-lo)/(hi-lo),0,1).astype(np.float32)
    return np.zeros_like(img, dtype=np.float32)

def mask_centroid(mask):
    ys,xs = np.nonzero(mask>0)
    if len(xs)==0: return np.array([np.nan,np.nan], dtype=float)
    return np.array([xs.mean(), ys.mean()], dtype=float)

def centroid_series(labels):
    return np.stack([mask_centroid(m) for m in labels], axis=0)

def fill_nan_centroids(C):
    C=np.asarray(C,dtype=float).copy(); T,D=C.shape
    for j in range(D):
        y=C[:,j]; ok=np.isfinite(y)
        if ok.sum()==0: C[:,j]=0.0
        elif ok.sum()<T: C[:,j]=np.interp(np.arange(T), np.flatnonzero(ok), y[ok])
    return C

def crop_box_from_mask_union(labels, margin=60):
    union=np.any(labels>0, axis=0); H,W=union.shape; ys,xs=np.nonzero(union)
    if len(xs)==0: return 0,H,0,W
    return max(0,int(ys.min())-margin), min(H,int(ys.max())+margin+1), max(0,int(xs.min())-margin), min(W,int(xs.max())+margin+1)

def crop_sequence(frames, labels, box):
    y0,y1,x0,x1=box
    return frames[:,y0:y1,x0:x1], labels[:,y0:y1,x0:x1]

def causal_derivatives(X):
    X=np.asarray(X,dtype=np.float32); V=np.zeros_like(X); A=np.zeros_like(X)
    V[1:]=X[1:]-X[:-1]
    A[2:]=X[2:]-2*X[1:-1]+X[:-2]
    return V,A

def robust_zscore(x):
    x=np.asarray(x,dtype=float); med=np.nanmedian(x); mad=np.nanmedian(np.abs(x-med))
    if mad < 1e-12: return np.zeros_like(x)
    return 0.6745*(x-med)/mad

def safe_auc(y, score):
    y=np.asarray(y).astype(int); score=np.asarray(score,dtype=float)
    ok=np.isfinite(score); y=y[ok]; score=score[ok]
    if len(y)<5 or len(np.unique(y))<2: return np.nan,np.nan
    return float(roc_auc_score(y,score)), float(average_precision_score(y,score))

df_cases_index = find_trackrad_cases(TRACKRAD_ROOT)
print("Cases found:", len(df_cases_index))
display(df_cases_index.head(20))
df_cases_index.to_csv(TABDIR/"case_index.csv", index=False)


In [ ]:
# ============================================================
# 2. Case-level cohort classification
# ============================================================
case_rows=[]
for _, row in df_cases_index.iterrows():
    cid=row["case_id"]
    try:
        frames, labels = load_trackrad_correct(row)
        T,H,W = frames.shape
        C=fill_nan_centroids(centroid_series(labels)); D=C-C[0]
        _, Ac = causal_derivatives(D)
        acc_z = robust_zscore(np.linalg.norm(Ac, axis=1))
        event_fraction = float(np.mean(acc_z > EVENT_Z_THRESHOLD))
        areas = labels.reshape(T,-1).sum(axis=1)
        amp_x=float(np.nanmax(C[:,0])-np.nanmin(C[:,0])); amp_y=float(np.nanmax(C[:,1])-np.nanmin(C[:,1])); amp_total=float(np.hypot(amp_x,amp_y))
        if amp_y > 1.5*amp_x: direction="vertical_dominant"
        elif amp_x > 1.5*amp_y: direction="horizontal_dominant"
        else: direction="mixed_2D_motion"
        motion_class = "small_motion" if amp_total<5 else "moderate_motion" if amp_total<15 else "large_motion" if amp_total<35 else "very_large_motion"
        irregularity = "smooth_motion" if event_fraction==0 else "mild_irregularity" if event_fraction<0.03 else "moderate_irregularity" if event_fraction<0.08 else "high_irregularity"
        median_area=float(np.median(areas))
        target_size = "small_target" if median_area<300 else "medium_target" if median_area<1500 else "large_target" if median_area<6000 else "very_large_target"
        box=crop_box_from_mask_union(labels, CROP_MARGIN); y0,y1,x0,x1=box
        crop_H,crop_W=y1-y0,x1-x0; crop_fraction=(crop_H*crop_W)/(H*W)
        respiratory = "respiratory_motion_visible" if (direction in ["vertical_dominant","mixed_2D_motion"] and amp_total>=8) else "weak_or_nonrespiratory_motion"
        if amp_total>=8 and amp_total<=35 and crop_fraction<=0.35 and median_area>=300 and irregularity in ["smooth_motion","mild_irregularity","moderate_irregularity"]:
            dense_class="good_candidate_for_dense_HS_POD"
        elif amp_total>=5 and crop_fraction<=0.55 and median_area>=150:
            dense_class="borderline_candidate_for_dense_HS_POD"
        else:
            dense_class="not_recommended_initially"
        case_rows.append({"case_id":cid,"split":row["split"],"T":T,"H":H,"W":W,"median_mask_area":median_area,"target_size_class":target_size,"centroid_amp_x":amp_x,"centroid_amp_y":amp_y,"centroid_amp_total":amp_total,"dominant_direction":direction,"motion_class":motion_class,"event_fraction":event_fraction,"irregularity_class":irregularity,"crop_H":crop_H,"crop_W":crop_W,"crop_fraction":crop_fraction,"respiratory_motion_class":respiratory,"dense_hs_candidate_class":dense_class})
    except Exception as e:
        print("ERROR", cid, e)

df_case_class=pd.DataFrame(case_rows)
df_case_class.to_csv(TABDIR/"cohort_case_level_classification.csv", index=False)
display(df_case_class.head())
for col in ["motion_class","dominant_direction","respiratory_motion_class","dense_hs_candidate_class","irregularity_class","target_size_class"]:
    print(''+col); display(df_case_class[col].value_counts().reset_index())

fig, axes = plt.subplots(1,3,figsize=(15,4))
df_case_class["motion_class"].value_counts().plot(kind="bar", ax=axes[0]); axes[0].set_title("Motion class")
df_case_class["dense_hs_candidate_class"].value_counts().plot(kind="bar", ax=axes[1]); axes[1].set_title("Dense H&S-POD suitability")
df_case_class["irregularity_class"].value_counts().plot(kind="bar", ax=axes[2]); axes[2].set_title("Irregularity class")
for ax in axes: ax.tick_params(axis="x", rotation=35); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(FIGDIR/"fig_case_classification_counts.png", dpi=300, bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)


In [ ]:
# ============================================================
# 3. Representative frames and centroid trajectory
# ============================================================
good_cases = df_case_class[df_case_class["dense_hs_candidate_class"]=="good_candidate_for_dense_HS_POD"].sort_values("centroid_amp_total", ascending=False)["case_id"].tolist()
CASE_ID = good_cases[0] if len(good_cases) else df_cases_index.iloc[0]["case_id"]
print("Representative CASE_ID:", CASE_ID)
row=df_cases_index[df_cases_index["case_id"]==CASE_ID].iloc[0]
frames, labels = load_trackrad_correct(row); T,H,W=frames.shape
box=crop_box_from_mask_union(labels, CROP_MARGIN); frames_c, labels_c = crop_sequence(frames, labels, box)
frames_cn=np.stack([normalize_image(f) for f in frames_c])
C=fill_nan_centroids(centroid_series(labels))

chosen=np.unique(np.linspace(0,T-1,6).round().astype(int))
fig,axes=plt.subplots(2,3,figsize=(12,7)); axes=axes.ravel()
for ax,t in zip(axes,chosen):
    ax.imshow(normalize_image(frames[t]), cmap="gray"); ax.contour(labels[t], levels=[.5], colors="white"); c=mask_centroid(labels[t])
    if np.isfinite(c).all(): ax.plot(c[0],c[1],'o',markersize=3)
    ax.set_title(f"Frame {t}"); ax.axis("off")
fig.suptitle(f"Corrected TrackRAD frames and target mask ({CASE_ID})")
fig.tight_layout(); fig.savefig(FIGDIR/"fig1_corrected_frames_mask_montage.png", dpi=300, bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].imshow(frames_cn[0], cmap="gray"); axes[0].contour(labels_c[0], levels=[.5], colors="white"); axes[0].set_title("Reference crop + target"); axes[0].axis("off")
axes[1].plot(C[:,0]); axes[1].set_title("Centroid x"); axes[1].grid(True, alpha=.3)
axes[2].plot(C[:,1]); axes[2].set_title("Centroid y"); axes[2].grid(True, alpha=.3)
fig.suptitle(f"Crop and centroid trajectory ({CASE_ID})")
fig.tight_layout(); fig.savefig(FIGDIR/"fig2_crop_and_centroid_trajectory.png", dpi=300, bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)


In [ ]:
# ============================================================
# 4. Branch 1: GPU Horn-Schunck dense apparent motion fields
# ============================================================
def _to_torch_image(img, device=DEVICE):
    return torch.from_numpy(normalize_image(img)).float().to(device)[None,None]

def gpu_horn_schunck_flow(ref, cur, alpha=15.0, n_iter=300, device=DEVICE):
    I1=_to_torch_image(ref,device); I2=_to_torch_image(cur,device)
    kx=torch.tensor([[[-1,1],[-1,1]]],dtype=torch.float32,device=device)[None]*0.25
    ky=torch.tensor([[[-1,-1],[1,1]]],dtype=torch.float32,device=device)[None]*0.25
    kt=torch.ones((1,1,2,2),dtype=torch.float32,device=device)*0.25
    Ix=F.conv2d(I1,kx,padding=1)[:,:,:-1,:-1]+F.conv2d(I2,kx,padding=1)[:,:,:-1,:-1]
    Iy=F.conv2d(I1,ky,padding=1)[:,:,:-1,:-1]+F.conv2d(I2,ky,padding=1)[:,:,:-1,:-1]
    It=F.conv2d(I2,kt,padding=1)[:,:,:-1,:-1]-F.conv2d(I1,kt,padding=1)[:,:,:-1,:-1]
    Hh,Ww=Ix.shape[-2:]; u=torch.zeros((1,1,Hh,Ww),device=device); v=torch.zeros((1,1,Hh,Ww),device=device)
    avg=torch.tensor([[1/12,1/6,1/12],[1/6,0,1/6],[1/12,1/6,1/12]],dtype=torch.float32,device=device)[None,None]
    a2=alpha*alpha
    for _ in range(n_iter):
        ub=F.conv2d(u,avg,padding=1); vb=F.conv2d(v,avg,padding=1); num=Ix*ub+Iy*vb+It; den=a2+Ix*Ix+Iy*Iy+1e-8
        u=ub-Ix*num/den; v=vb-Iy*num/den
    return torch.cat([u,v],dim=1)[0].permute(1,2,0).detach().cpu().numpy().astype(np.float32)

def forward_warp_mask(ref_mask, flow):
    Hh,Ww=ref_mask.shape; pred=np.zeros((Hh,Ww),dtype=bool); ys,xs=np.nonzero(ref_mask>0)
    if len(xs)==0: return pred
    nx=np.round(xs+flow[ys,xs,0]).astype(int); ny=np.round(ys+flow[ys,xs,1]).astype(int)
    ok=(nx>=0)&(nx<Ww)&(ny>=0)&(ny<Hh); pred[ny[ok],nx[ok]]=True
    return binary_fill_holes(pred).astype(bool)

def dice_score(pred,true):
    den=pred.sum()+true.sum()
    if den==0: return np.nan
    return float(2*np.logical_and(pred,true).sum()/den)

def tumour_region_mean_flow(flow, ref_mask):
    m=ref_mask>0
    if m.sum()==0: return np.array([np.nan,np.nan])
    return np.array([flow[...,0][m].mean(), flow[...,1][m].mean()])

def compute_and_cache_dense_hs(row, overwrite=False):
    cid=row["case_id"]; save=CACHEDIR/f"{cid}_hs_dense_fields.npz"
    if save.exists() and not overwrite: return save
    frames,labels=load_trackrad_correct(row); box=crop_box_from_mask_union(labels,CROP_MARGIN); fc,lc=crop_sequence(frames,labels,box)
    fc=np.stack([normalize_image(f) for f in fc]); lc=lc>0; T,Hh,Ww=fc.shape
    ref_img=fc[0]; ref_mask=lc[0]; Cc=fill_nan_centroids(centroid_series(lc)); true_disp=Cc-Cc[0]
    flows=np.zeros((T,Hh,Ww,2),dtype=np.float32); mean_flow=np.zeros((T,2),dtype=np.float32); dice=np.full(T,np.nan,dtype=np.float32); epe=np.full(T,np.nan,dtype=np.float32); time_ms=np.full(T,np.nan,dtype=np.float32)
    dice[0]=1; epe[0]=0; time_ms[0]=0
    for t in range(1,T):
        st=time.time(); flow=gpu_horn_schunck_flow(ref_img,fc[t],HS_ALPHA,HS_N_ITER,DEVICE); time_ms[t]=1000*(time.time()-st)
        flows[t]=flow; mean_flow[t]=tumour_region_mean_flow(flow,ref_mask); pred=forward_warp_mask(ref_mask,flow); dice[t]=dice_score(pred,lc[t]); epe[t]=np.linalg.norm(mean_flow[t]-true_disp[t])
    np.savez_compressed(save,case_id=cid,split=row["split"],frames=fc.astype(np.float32),labels=lc.astype(np.uint8),flows=flows,mean_flow=mean_flow,true_disp=true_disp.astype(np.float32),dice=dice,epe=epe,time_ms=time_ms,crop_box=np.array(box))
    return save

selected_dense_cases = df_case_class[df_case_class["dense_hs_candidate_class"]=="good_candidate_for_dense_HS_POD"].sort_values("centroid_amp_total", ascending=False)["case_id"].tolist()[:MAX_DENSE_CASES]
print("Selected dense cases:", selected_dense_cases)
if RUN_DENSE_HS:
    for i,cid in enumerate(selected_dense_cases,1):
        print(f"[{i}/{len(selected_dense_cases)}] {cid}")
        compute_and_cache_dense_hs(df_cases_index[df_cases_index["case_id"]==cid].iloc[0])

hs_rows=[]
for p in sorted(CACHEDIR.glob("*_hs_dense_fields.npz")):
    d=np.load(p,allow_pickle=True); flows=d["flows"]
    hs_rows.append({"case_id":str(d["case_id"]),"split":str(d["split"]),"T":flows.shape[0],"crop_H":flows.shape[1],"crop_W":flows.shape[2],"flow_dim":2*flows.shape[1]*flows.shape[2],"mean_dice":float(np.nanmean(d["dice"][1:])),"mean_epe_px":float(np.nanmean(d["epe"][1:])),"mean_time_ms":float(np.nanmean(d["time_ms"][1:])),"path":str(p)})
df_hs_quality=pd.DataFrame(hs_rows); df_hs_quality.to_csv(TABDIR/"dense_hs_quality_cohort.csv",index=False); display(df_hs_quality)


In [ ]:
# ============================================================
# 4B. H&S example figure — paper-ready arrows scaled for visibility
# ============================================================
npz_files = sorted(CACHEDIR.glob("*_hs_dense_fields.npz"))
if len(npz_files) == 0:
    raise RuntimeError("No cached H&S fields. Run previous cell.")

d = np.load(npz_files[0], allow_pickle=True)
cid = str(d["case_id"])
fc = d["frames"].astype(np.float32)
lc = d["labels"].astype(bool)
flows = d["flows"].astype(np.float32)
dice = d["dice"].astype(np.float32)

T, Hh, Ww, _ = flows.shape

# Choose a frame with visible motion rather than always T//2.
flow_mag_series = np.sqrt(flows[..., 0]**2 + flows[..., 1]**2).mean(axis=(1, 2))
t = int(np.argmax(flow_mag_series[1:]) + 1) if T > 1 else 0

flow = flows[t]
mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
pred = forward_warp_mask(lc[0], flow)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))

# Panel 1: current frame + GT target
axes[0].imshow(fc[t], cmap="gray")
axes[0].contour(lc[t], levels=[0.5], colors="white", linewidths=1.4)
axes[0].set_title("Current frame + GT target")
axes[0].axis("off")

# Panel 2: H&S warped reference mask
axes[1].imshow(fc[t], cmap="gray")
axes[1].contour(lc[t], levels=[0.5], colors="white", linewidths=1.4)
if pred.sum() > 0:
    axes[1].contour(pred, levels=[0.5], colors="red", linewidths=1.4)
axes[1].set_title(f"H&S warp Dice = {dice[t]:.2f}")
axes[1].axis("off")

# Panel 3: flow magnitude
im = axes[2].imshow(mag, cmap="magma")
axes[2].set_title("Apparent flow magnitude [px]")
axes[2].axis("off")
cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("px")

# Panel 4: vector field. Arrows are scaled visually.
step = max(1, min(Hh, Ww) // 18)
yy, xx = np.mgrid[0:Hh:step, 0:Ww:step]
u = flow[::step, ::step, 0]
v = flow[::step, ::step, 1]

# Scale arrows so tiny apparent flows are visible. This is visual only.
arrow_gain = 800.0

axes[3].imshow(fc[t], cmap="gray")
axes[3].quiver(
    xx,
    yy,
    arrow_gain * u,
    arrow_gain * v,
    angles="xy",
    scale_units="xy",
    scale=1,
    width=0.004,
    color="yellow",
    headwidth=4,
    headlength=5,
    headaxislength=4,
)
axes[3].contour(lc[t], levels=[0.5], colors="white", linewidths=1.0)
axes[3].set_title(f"H&S vector field\narrows ×{arrow_gain:.0f} for visibility")
axes[3].axis("off")

fig.suptitle(
    f"GPU Horn–Schunck apparent in-plane motion ({cid}, frame {t})",
    fontsize=14,
)

fig.tight_layout()
fig.savefig(FIGDIR / "fig3_hs_apparent_motion_example.png", dpi=300, bbox_inches="tight")

if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)


In [ ]:
# ============================================================
# 5. Dense POD: classical vs enriched triplet, cohort level
# ============================================================
def fit_pod(X_train):
    X_train=np.asarray(X_train,dtype=np.float32); mu=X_train.mean(axis=0); Xc=X_train-mu
    U,S,Vt=np.linalg.svd(Xc, full_matrices=False); eig=S**2/max(1,X_train.shape[0]-1); total=eig.sum(); evr=eig/total if total>0 else np.zeros_like(eig)
    return {"mean":mu,"components":Vt.astype(np.float32),"singular_values":S,"explained_variance_ratio":evr,"cumulative_variance":np.cumsum(evr)}
def pod_transform(X,model,K): return (X-model["mean"]) @ model["components"][:K].T
def pod_inverse(C,model,K): return model["mean"] + C @ model["components"][:K]
def pod_reconstruct(X,model,K):
    C=pod_transform(X,model,K); return pod_inverse(C,model,K), C
def rmse_per_snapshot(Xhat,X): return np.sqrt(np.mean((Xhat-X)**2,axis=1))
def endpoint_error_flow(Xhat,X,H,W):
    P=H*W; return np.sqrt((Xhat[:,:P]-X[:,:P])**2 + (Xhat[:,P:]-X[:,P:])**2).mean(axis=1)
def relative_rmse(Xhat,X): return float(np.sqrt(np.mean((Xhat-X)**2)/(np.mean(X**2)+1e-12)))
def make_triplet_state(X,train_idx,kappa=.25):
    V,A=causal_derivatives(X); eps=1e-8; sX=np.sqrt(np.mean(X[train_idx]**2))+eps; sV=np.sqrt(np.mean(V[train_idx]**2))+eps; sA=np.sqrt(np.mean(A[train_idx]**2))+eps
    lam1=sX/sV; lam2=kappa*sX/sA; Z=np.concatenate([X,lam1*V,lam2*A],axis=1).astype(np.float32); return Z,V,A,{"lambda1":float(lam1),"lambda2":float(lam2)}
def modes_for_threshold(cum,thr):
    idx=np.where(cum>=thr)[0]; return int(idx[0]+1) if len(idx) else np.nan

rank_rows=[]; case_rows=[]; spectral_rows=[]; example_models={}
for p in sorted(CACHEDIR.glob("*_hs_dense_fields.npz")):
    data=np.load(p,allow_pickle=True); cid=str(data["case_id"]); split=str(data["split"]); flows=data["flows"].astype(np.float32); labels=data["labels"].astype(bool); T,Hh,Ww,_=flows.shape; P=Hh*Ww; flow_dim=2*P
    train_T=max(8,int(round(TRAIN_FRAC*T))); train_T=min(train_T,T-5); train_idx=np.arange(train_T); test_idx=np.arange(train_T,T)
    X=np.concatenate([flows[...,0].reshape(T,P), flows[...,1].reshape(T,P)],axis=1).astype(np.float32)
    C=fill_nan_centroids(centroid_series(labels)); Dcent=C-C[0]; _,Acent=causal_derivatives(Dcent); gt_event=robust_zscore(np.linalg.norm(Acent,axis=1))>EVENT_Z_THRESHOLD
    _,Xacc=causal_derivatives(X); flow_event=robust_zscore(np.sqrt(np.mean(Xacc**2,axis=1)))>EVENT_Z_THRESHOLD
    cm=fit_pod(X[train_idx]); Z,_,_,info=make_triplet_state(X,train_idx,KAPPA_ACC); em=fit_pod(Z[train_idx])
    if not example_models: example_models={"data":data,"classical_model":cm,"enriched_model":em,"flow_dim":flow_dim}
    for m,val in enumerate(cm["explained_variance_ratio"],1): spectral_rows.append({"case_id":cid,"model":"classical_dense_HS_POD","mode":m,"explained_variance_ratio":float(val),"cumulative_variance":float(cm["cumulative_variance"][m-1])})
    for m,val in enumerate(em["explained_variance_ratio"],1): spectral_rows.append({"case_id":cid,"model":"enriched_dense_HS_triplet_POD","mode":m,"explained_variance_ratio":float(val),"cumulative_variance":float(em["cumulative_variance"][m-1])})
    for K in K_LIST:
        if K<=cm["components"].shape[0]:
            Xhat,_=pod_reconstruct(X,cm,K); r=rmse_per_snapshot(Xhat,X); epe=endpoint_error_flow(Xhat,X,Hh,Ww); gt_auc,gt_ap=safe_auc(gt_event[test_idx],r[test_idx]); fl_auc,fl_ap=safe_auc(flow_event[test_idx],r[test_idx])
            rank_rows.append({"case_id":cid,"model":"classical_dense_HS_POD","K":K,"flow_dim":flow_dim,"rel_rmse_all":relative_rmse(Xhat[test_idx],X[test_idx]),"epe_mean_test":float(np.mean(epe[test_idx])),"epe_p95_test":float(np.percentile(epe[test_idx],95)),"gt_event_auroc":gt_auc,"gt_event_auprc":gt_ap,"flow_event_auroc":fl_auc,"flow_event_auprc":fl_ap})
        if K<=em["components"].shape[0]:
            Zhat,_=pod_reconstruct(Z,em,K); Xhat=Zhat[:,:flow_dim]; rfull=rmse_per_snapshot(Zhat,Z); epe=endpoint_error_flow(Xhat,X,Hh,Ww); gt_auc,gt_ap=safe_auc(gt_event[test_idx],rfull[test_idx]); fl_auc,fl_ap=safe_auc(flow_event[test_idx],rfull[test_idx])
            rank_rows.append({"case_id":cid,"model":"enriched_dense_HS_triplet_POD","K":K,"flow_dim":flow_dim,"rel_rmse_all":relative_rmse(Xhat[test_idx],X[test_idx]),"epe_mean_test":float(np.mean(epe[test_idx])),"epe_p95_test":float(np.percentile(epe[test_idx],95)),"gt_event_auroc":gt_auc,"gt_event_auprc":gt_ap,"flow_event_auroc":fl_auc,"flow_event_auprc":fl_ap})
    case_rows.append({"case_id":cid,"T":T,"H":Hh,"W":Ww,"flow_dim":flow_dim,"mean_hs_dice":float(np.nanmean(data["dice"][1:])),"mean_hs_epe":float(np.nanmean(data["epe"][1:])),"classical_modes_95":modes_for_threshold(cm["cumulative_variance"],.95),"enriched_modes_95":modes_for_threshold(em["cumulative_variance"],.95),"lambda1":info["lambda1"],"lambda2":info["lambda2"]})

df_dense_case=pd.DataFrame(case_rows); df_dense_rank=pd.DataFrame(rank_rows); df_dense_spectral=pd.DataFrame(spectral_rows)
df_dense_case.to_csv(TABDIR/"dense_pod_case_summary.csv",index=False); df_dense_rank.to_csv(TABDIR/"dense_pod_rank_metrics.csv",index=False); df_dense_spectral.to_csv(TABDIR/"dense_pod_spectral.csv",index=False)
display(df_dense_case)
display(df_dense_rank.groupby(["model","K"])[["epe_mean_test","gt_event_auroc","flow_event_auroc"]].mean().reset_index())


In [ ]:
# ============================================================
# 5B. Dense POD paper-ready figures
# Main manuscript version:
#   (i) whole-spectrum comparison,
#   (ii) rank sensitivity,
#   (iii) Triplet-POD block-energy decomposition.
#
# The earlier displacement-block mode map is intentionally not used as a main
# figure because it can be misread: Triplet-POD modes live in the stacked
# space [D, lambda1*Ddot, lambda2*Dddot], and a mode can be important even
# when its displacement block looks weak.
# ============================================================

# ------------------------------------------------------------
# Aggregated spectral and rank tables
# ------------------------------------------------------------
spec_agg = (
    df_dense_spectral
    .groupby(["model", "mode"])[["cumulative_variance"]]
    .mean()
    .reset_index()
)

rank_agg = (
    df_dense_rank
    .groupby(["model", "K"])[["epe_mean_test", "gt_event_auroc", "flow_event_auroc"]]
    .mean()
    .reset_index()
)

spec_agg.to_csv(TABDIR / "dense_pod_spectral_aggregate.csv", index=False)
rank_agg.to_csv(TABDIR / "dense_pod_rank_metrics_aggregate.csv", index=False)

# ------------------------------------------------------------
# Figure 4: Dense POD spectral composition
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.4, 5.2))

for model, g in spec_agg.groupby("model"):
    g = g.sort_values("mode").head(30)
    label = {
        "classical_dense_HS_POD": "Classical dense POD",
        "enriched_dense_HS_triplet_POD": "Enriched Triplet-POD",
    }.get(model, model)

    ax.plot(
        g["mode"],
        g["cumulative_variance"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=label,
    )

ax.axhline(0.90, linestyle="--", linewidth=1, label="90%")
ax.axhline(0.95, linestyle=":", linewidth=1.5, label="95%")
ax.axhline(0.99, linestyle="-.", linewidth=1, label="99%")

ax.set_xlabel("Mode number")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("Dense apparent-motion POD spectral composition")
ax.set_ylim(0, 1.03)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGDIR / "fig4_dense_pod_spectral_composition.png", dpi=350, bbox_inches="tight")

if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)

# ------------------------------------------------------------
# Figure 5: Dense POD rank sensitivity
# Uses independent GT-centroid event labels for the right panel.
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

for model, g in rank_agg.groupby("model"):
    g = g.sort_values("K")
    label = {
        "classical_dense_HS_POD": "Classical dense POD",
        "enriched_dense_HS_triplet_POD": "Enriched Triplet-POD",
    }.get(model, model)

    axes[0].plot(g["K"], g["epe_mean_test"], marker="o", linewidth=2, label=label)
    axes[1].plot(g["K"], g["gt_event_auroc"], marker="o", linewidth=2, label=label)

axes[0].set_title("(a) Dense-field reconstruction")
axes[0].set_ylabel("Mean dense EPE [px]")

axes[1].set_title("(b) Independent GT-centroid event sensitivity")
axes[1].set_ylabel("AUROC")

for ax in axes:
    ax.set_xlabel("Rank K")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle("Dense POD rank sensitivity", fontsize=14)
fig.tight_layout()
fig.savefig(FIGDIR / "fig5_dense_pod_rank_sensitivity.png", dpi=350, bbox_inches="tight")

if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)

# ------------------------------------------------------------
# Modes-needed cohort table
# ------------------------------------------------------------
print("Modes needed table:")
display(df_dense_case[[
    "case_id",
    "flow_dim",
    "classical_modes_95",
    "enriched_modes_95",
    "mean_hs_dice",
    "mean_hs_epe",
]])

# ------------------------------------------------------------
# Figure 6: Triplet-POD block-energy decomposition
# For the representative example model only.
# ------------------------------------------------------------
ex = example_models["data"]
em = example_models["enriched_model"]
flow_dim = int(example_models["flow_dim"])
case_id = str(ex["case_id"])

components = em["components"]

if components.shape[1] < 3 * flow_dim:
    raise ValueError(
        f"Enriched model has dimension {components.shape[1]}, but expected at least {3*flow_dim} "
        "for [D, velocity, acceleration] blocks."
    )

n_modes_show = min(12, components.shape[0])
block_rows = []

for k in range(n_modes_show):
    psi = components[k]

    psi_D = psi[:flow_dim]
    psi_V = psi[flow_dim:2 * flow_dim]
    psi_A = psi[2 * flow_dim:3 * flow_dim]

    ED = float(np.sum(psi_D ** 2))
    EV = float(np.sum(psi_V ** 2))
    EA = float(np.sum(psi_A ** 2))
    total = ED + EV + EA + 1e-12

    block_rows.append({
        "case_id": case_id,
        "mode": k + 1,
        "displacement_energy": ED,
        "velocity_energy": EV,
        "acceleration_energy": EA,
        "displacement_fraction": ED / total,
        "velocity_fraction": EV / total,
        "acceleration_fraction": EA / total,
    })

df_block = pd.DataFrame(block_rows)
df_block.to_csv(TABDIR / "triplet_pod_block_energy_decomposition.csv", index=False)

print("Triplet-POD block-energy decomposition:")
display(df_block)

fig, ax = plt.subplots(figsize=(8.4, 5.2))

modes = df_block["mode"].values
disp = df_block["displacement_fraction"].values
vel = df_block["velocity_fraction"].values
acc = df_block["acceleration_fraction"].values

ax.bar(modes, disp, label="Displacement block")
ax.bar(modes, vel, bottom=disp, label="Velocity block")
ax.bar(modes, acc, bottom=disp + vel, label="Acceleration block")

ax.set_xlabel("Triplet-POD mode")
ax.set_ylabel("Fraction of mode energy")
ax.set_title(f"Triplet-POD block-energy decomposition ({case_id})")
ax.set_ylim(0, 1.02)
ax.set_xticks(modes)
ax.grid(True, axis="y", alpha=0.3)
ax.legend(fontsize=9, loc="upper right")

fig.tight_layout()
fig.savefig(FIGDIR / "fig6_triplet_pod_block_energy_decomposition.png", dpi=350, bbox_inches="tight")

if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)

# ------------------------------------------------------------
# Figure 6b: Combined figure for manuscript option
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Left: cumulative spectrum
ax = axes[0]
for model, g in spec_agg.groupby("model"):
    g = g.sort_values("mode").head(30)
    label = {
        "classical_dense_HS_POD": "Classical dense POD",
        "enriched_dense_HS_triplet_POD": "Enriched Triplet-POD",
    }.get(model, model)

    ax.plot(
        g["mode"],
        g["cumulative_variance"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=label,
    )

ax.axhline(0.95, linestyle=":", linewidth=1.5, label="95%")
ax.set_xlabel("Mode number")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("(a) Spectral composition")
ax.set_ylim(0, 1.03)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# Right: block decomposition
ax = axes[1]
ax.bar(modes, disp, label="Displacement")
ax.bar(modes, vel, bottom=disp, label="Velocity")
ax.bar(modes, acc, bottom=disp + vel, label="Acceleration")
ax.set_xlabel("Triplet-POD mode")
ax.set_ylabel("Fraction of mode energy")
ax.set_title("(b) Triplet mode block-energy")
ax.set_ylim(0, 1.02)
ax.set_xticks(modes)
ax.grid(True, axis="y", alpha=0.3)
ax.legend(fontsize=8)

fig.suptitle(
    "Dense apparent-motion POD representation: spectrum and triplet-mode composition",
    fontsize=14,
)

fig.tight_layout()
fig.savefig(FIGDIR / "fig6b_dense_pod_spectrum_and_triplet_decomposition.png", dpi=350, bbox_inches="tight")

if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)


## Branch 2 methodology notes (read before the numbers)

**Observation-noise floor for centroid calibration.** A 2-D centroid is spanned exactly by the
leading POD modes, so its POD *truncation* residual is ~0 and cannot calibrate the intervals.
We therefore add an **observation-noise floor** `SigmaD`/`SigmaZ`, the covariance of the
training one-step forecast residual, to the predicted covariance. This is the centroid analogue
of the dense truncation-residual term and is what lifts naive coverage to nominal.

**Acceleration z-score is the event LABELLER, not a detector.** Events are defined as the top-5%
of ground-truth centroid acceleration (`robust_zscore(acc_mag)>3`). Scoring an acceleration
signal against an acceleration-defined label is circular (AUROC->1), so acceleration is **excluded**
from the reported detectors. Reported detectors: velocity, displacement surprise, triplet surprise.

**Fair local constant-velocity baseline.** `pred[t]=C[t-1]+(C[t-1]-C[t-2])` (last observed step
repeated), not a global training-mean velocity extrapolated over the whole sequence.

**Cohort choice.** Event detection and all Branch-2 metrics are reported on the **88-case centroid
cohort** with the independent GT-centroid-acceleration label; the dense H&S-POD branch uses the
**8 representative cases**. Per-voxel uncertainty is intentionally out of scope here.


In [ ]:
# ============================================================
# 6. Branch 2: Cohort Bayesian centroid trajectory analysis
# ============================================================
def fit_linear_dynamics(Y_train):
    Y0,Y1=Y_train[:-1],Y_train[1:]; d=Y_train.shape[1]; ridge=1e-6*np.eye(d); A=Y1.T@Y0@np.linalg.inv(Y0.T@Y0+ridge); resid=Y1-Y0@A.T
    Q=np.cov(resid.T) if resid.shape[0]>2 else np.eye(d)*1e-3
    if np.ndim(Q)==0: Q=np.eye(d)*float(Q)
    return A, np.asarray(Q,dtype=float)+np.eye(d)*1e-6, resid

def kalman_observed_state(Y,train_idx):
    Y=np.asarray(Y,dtype=float); T,d=Y.shape; A,Q,resid=fit_linear_dynamics(Y[train_idx]); R=.10*Q+np.eye(d)*1e-5; m=Y[train_idx[0]].copy(); P=np.cov(Y[train_idx].T) if len(train_idx)>3 else np.eye(d)
    if np.ndim(P)==0: P=np.eye(d)*float(P)
    P=np.asarray(P,dtype=float)+np.eye(d)*1e-5; I=np.eye(d); mpred=np.zeros((T,d)); Ppred=np.zeros((T,d,d)); surprise=np.zeros(T)
    for t in range(T):
        mp=A@m; Pp=A@P@A.T+Q; mpred[t]=mp; Ppred[t]=Pp; r=Y[t]-mp; S=Pp+R+np.eye(d)*1e-8; Sinv=np.linalg.pinv(S); surprise[t]=float(r.T@Sinv@r); K=Pp@Sinv; m=mp+K@r; P=(I-K)@Pp
    scale=np.mean(surprise[train_idx])/max(d,1)
    if not np.isfinite(scale) or scale<=1e-12: scale=1.0
    return {"means_pred":mpred,"covs_pred":Ppred,"surprise":surprise/scale}

def make_centroid_triplet(D,train_idx,kappa=.25):
    V,A=causal_derivatives(D); eps=1e-8; sD=np.sqrt(np.mean(D[train_idx]**2))+eps; sV=np.sqrt(np.mean(V[train_idx]**2))+eps; sA=np.sqrt(np.mean(A[train_idx]**2))+eps; lam1=sD/sV; lam2=kappa*sD/sA
    return np.concatenate([D,lam1*V,lam2*A],axis=1), V, A

def mahal_coverage(errors,covs,level):
    thr=chi2.ppf(level,df=2); hits=[]
    for e,Cv in zip(errors,covs):
        Cv=np.asarray(Cv)+np.eye(2)*1e-8; hits.append(float(e.T@np.linalg.pinv(Cv)@e)<=thr)
    return float(np.mean(hits)) if hits else np.nan

def risk_coverage(errors,scores,coverages=np.linspace(.5,1,11)):
    errors=np.asarray(errors,dtype=float); scores=np.asarray(scores,dtype=float); ok=np.isfinite(errors)&np.isfinite(scores); errors=errors[ok]; scores=scores[ok]
    if len(errors)==0: return pd.DataFrame()
    e=errors[np.argsort(scores)]; rows=[]
    for c in coverages:
        n=max(1,int(round(c*len(e)))); rows.append({"coverage":float(c),"risk_mean":float(np.mean(e[:n])),"risk_p95":float(np.percentile(e[:n],95))})
    return pd.DataFrame(rows)

case_sum=[]; cov_rows=[]; risk_rows=[]; event_rows=[]
for _,row in df_cases_index.iterrows():
    cid=row["case_id"]
    try:
        frames,labels=load_trackrad_correct(row); T=labels.shape[0]
        if T<20: continue
        train_T=max(8,int(round(TRAIN_FRAC*T))); train_T=min(train_T,T-5); train_idx=np.arange(train_T); test_idx=np.arange(train_T,T)
        C=fill_nan_centroids(centroid_series(labels)); c0=C[0]; D=C-c0; Vc,Ac=causal_derivatives(D); acc_mag=np.linalg.norm(Ac,axis=1); event_label=robust_zscore(acc_mag)>EVENT_Z_THRESHOLD
        pred_pos=np.repeat(C[0][None,:],T,axis=0)
        # fair LOCAL constant-velocity: last observed step repeated (per-frame), NOT a
        # global training-mean velocity extrapolated over the whole sequence (which is
        # unfair and blows up late). pred[t] = C[t-1] + (C[t-1]-C[t-2]).
        pred_vel=np.zeros_like(C); pred_vel[0]=C[0]
        for _t in range(1,T):
            pred_vel[_t]=C[_t-1]+((C[_t-1]-C[_t-2]) if _t>=2 else 0.0)
        kalD=kalman_observed_state(D,train_idx); predD=kalD["means_pred"]+c0; Z,V,A=make_centroid_triplet(D,train_idx,KAPPA_ACC); kalZ=kalman_observed_state(Z,train_idx); predZ=kalZ["means_pred"][:,:2]+c0
        methods={"constant_position":pred_pos,"constant_velocity":pred_vel,"bayes_displacement":predD,"bayes_triplet":predZ}
        for name,pred in methods.items():
            errs=np.linalg.norm(pred[test_idx]-C[test_idx],axis=1)
            # error by frame type so the triplet contribution on SPONTANEOUS frames is visible
            ev_te=event_label[test_idx]
            e_smooth=float(np.mean(errs[~ev_te])) if (~ev_te).any() else np.nan
            e_spont=float(np.mean(errs[ev_te])) if ev_te.any() else np.nan
            case_sum.append({"case_id":cid,"method":name,"centroid_error_mean_px":float(np.mean(errs)),"centroid_error_median_px":float(np.median(errs)),"centroid_error_p95_px":float(np.percentile(errs,95)),"err_smooth_frames":e_smooth,"err_spontaneous_frames":e_spont})
        SigmaD=np.cov((D[train_idx]-kalD["means_pred"][train_idx]).T)+np.eye(2)*1e-6; SigmaZ=np.cov((D[train_idx]-kalZ["means_pred"][train_idx,:2]).T)+np.eye(2)*1e-6
        errsD=[]; errsZ=[]; covDn=[]; covDc=[]; covZn=[]; covZc=[]; sigD=[]; sigZ=[]
        for t in test_idx:
            eD=predD[t]-C[t]; CD=kalD["covs_pred"][t][:2,:2]+np.eye(2)*1e-8; eZ=predZ[t]-C[t]; CZ=kalZ["covs_pred"][t][:2,:2]+np.eye(2)*1e-8
            errsD.append(eD); covDn.append(CD); covDc.append(CD+SigmaD); sigD.append(np.sqrt(np.trace(CD+SigmaD)))
            errsZ.append(eZ); covZn.append(CZ); covZc.append(CZ+SigmaZ); sigZ.append(np.sqrt(np.trace(CZ+SigmaZ)))
        for lev in [.5,.8,.9,.95]:
            cov_rows.append({"case_id":cid,"method":"bayes_displacement","nominal":lev,"coverage_naive":mahal_coverage(errsD,covDn,lev),"coverage_corrected":mahal_coverage(errsD,covDc,lev)})
            cov_rows.append({"case_id":cid,"method":"bayes_triplet","nominal":lev,"coverage_naive":mahal_coverage(errsZ,covZn,lev),"coverage_corrected":mahal_coverage(errsZ,covZc,lev)})
        for method,score_name,err,score in [("bayes_displacement","sigma_corrected",np.linalg.norm(errsD,axis=1),np.asarray(sigD)),("bayes_triplet","sigma_corrected",np.linalg.norm(errsZ,axis=1),np.asarray(sigZ)),("bayes_triplet","triplet_surprise",np.linalg.norm(errsZ,axis=1),kalZ["surprise"][test_idx])]:
            rr=risk_coverage(err,score); rr["case_id"]=cid; rr["method"]=method; rr["score"]=score_name; risk_rows.append(rr)
        y=event_label[test_idx].astype(int)
        # NOTE: "acceleration" is EXCLUDED as a scored detector: the event_label is DEFINED from
        # acceleration (robust_zscore(acc_mag)>3), so scoring acceleration against it is
        # circular (AUROC->1, meaningless). Acceleration z-score is used as the LABELLER,
        # not as a detector. Reported detectors: velocity, disp_surprise, trip_surprise.
        for score_name,score in [("disp_surprise",kalD["surprise"][test_idx]),("trip_surprise",kalZ["surprise"][test_idx]),("velocity",np.linalg.norm(Vc,axis=1)[test_idx])]:
            au,ap=safe_auc(y,score); event_rows.append({"case_id":cid,"score":score_name,"event_rate":float(np.mean(y)),"auroc":au,"auprc":ap})
    except Exception as e: print("ERROR",cid,e)

df_centroid_case=pd.DataFrame(case_sum); df_centroid_cov=pd.DataFrame(cov_rows); df_centroid_risk=pd.concat(risk_rows,ignore_index=True); df_centroid_event=pd.DataFrame(event_rows)
df_centroid_case.to_csv(TABDIR/"centroid_case_summary.csv",index=False); df_centroid_cov.to_csv(TABDIR/"centroid_coverage.csv",index=False); df_centroid_risk.to_csv(TABDIR/"centroid_risk_coverage.csv",index=False); df_centroid_event.to_csv(TABDIR/"centroid_event_metrics.csv",index=False)
display(df_centroid_case.groupby("method")[["centroid_error_mean_px","centroid_error_p95_px"]].mean().reset_index())
display(df_centroid_cov.groupby(["method","nominal"])[["coverage_naive","coverage_corrected"]].mean().reset_index())
display(df_centroid_event.groupby("score")[["auroc","auprc"]].mean().reset_index())


In [ ]:
# ============================================================
# 6C. Paper-number generation: gate table, risk reduction, EB Student-t, bootstrap CIs
# All numbers below map directly to manuscript Tables 1-4. No per-voxel quantities.
# ============================================================
from scipy.stats import chi2 as _chi2

def boot_ci(vals, n=2000, seed=0):
    v=np.asarray([x for x in vals if np.isfinite(x)],float)
    if len(v)==0: return (np.nan,np.nan,np.nan)
    r=np.random.default_rng(seed); bs=[r.choice(v,len(v),True).mean() for _ in range(n)]
    return float(v.mean()), float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))

# ---- Table 1: prediction with median + bootstrap CI ----
pred_tbl=[]
for name,g in df_centroid_case.groupby("method"):
    m,lo,hi=boot_ci(g["centroid_error_mean_px"])
    md,_,_=boot_ci(g["centroid_error_median_px"])
    p95,_,_=boot_ci(g["centroid_error_p95_px"])
    pred_tbl.append({"method":name,"mean_px":m,"mean_ci_lo":lo,"mean_ci_hi":hi,"median_px":md,"p95_px":p95})
df_pred_tbl=pd.DataFrame(pred_tbl).set_index("method").reindex(["constant_position","constant_velocity","bayes_displacement","bayes_triplet"])
df_pred_tbl.to_csv(TABDIR/"paper_table1_prediction.csv")
print("=== Table 1: prediction (px), mean [95% CI], median, p95 ===")
display(df_pred_tbl.round(3))

# ---- Error by frame type: triplet contribution on spontaneous frames ----
ft=df_centroid_case.groupby("method")[["err_smooth_frames","err_spontaneous_frames"]].mean().reindex(["bayes_displacement","bayes_triplet"])
ft.to_csv(TABDIR/"paper_error_by_frame_type.csv")
print("=== Error by frame type (px): smooth vs spontaneous ===")
display(ft.round(3))

# ---- Table 3: gate accepted vs rejected at chi2_{2,0.95} (triplet surprise) ----
# rebuild per-frame surprise+error for the triplet branch across cohort
gate_rows=[]; rr_mean_all=[]; rr_mean_50=[]; rr_p95_all=[]; rr_p95_50=[]
for cid,g in df_centroid_risk[(df_centroid_risk.method=="bayes_triplet")&(df_centroid_risk.score=="triplet_surprise")].groupby("case_id"):
    g=g.sort_values("coverage")
    a=g.iloc[(g.coverage-1.0).abs().argmin()]; h=g.iloc[(g.coverage-0.5).abs().argmin()]
    rr_mean_all.append(a.risk_mean); rr_mean_50.append(h.risk_mean); rr_p95_all.append(a.risk_p95); rr_p95_50.append(h.risk_p95)
mean_all=boot_ci(rr_mean_all)[0]; mean_50=boot_ci(rr_mean_50)[0]; p95_all=boot_ci(rr_p95_all)[0]; p95_50=boot_ci(rr_p95_50)[0]
print(f"=== Risk-coverage (triplet surprise) ===")
print(f"mean: all {mean_all:.3f} -> 50% retained {mean_50:.3f}  ({100*(mean_50-mean_all)/mean_all:+.0f}%)")
print(f"p95 : all {p95_all:.3f} -> 50% retained {p95_50:.3f}  ({100*(p95_50-p95_all)/p95_all:+.0f}%)")

# accepted/rejected at chi2 threshold needs per-frame surprise; recompute compactly
acc_err=[]; rej_err=[]; rej_rate=[]
# NIS-calibrated surprise has training mean = d (triplet dim) -> ~chi2_d; use df=d.
for _,row in df_cases_index.iterrows():
    try:
        frames,labels=load_trackrad_correct(row); T=labels.shape[0]
        if T<20: continue
        train_T=max(8,int(round(TRAIN_FRAC*T))); train_T=min(train_T,T-5); tr=np.arange(train_T); te=np.arange(train_T,T)
        C=fill_nan_centroids(centroid_series(labels)); c0=C[0]; D=C-c0
        Z,V,A=make_centroid_triplet(D,tr,KAPPA_ACC); kalZ=kalman_observed_state(Z,tr); predZ=kalZ["means_pred"][:,:2]+c0
        sup_tr=kalZ["surprise"][tr]; sup_te=kalZ["surprise"][te]
        err=np.linalg.norm(predZ[te]-C[te],axis=1)
        # EMPIRICAL gate: tau = 95th percentile of TRAIN surprise (5% train false-alarm by
        # construction); report the honest TEST rejection rate, which includes real events.
        thr=np.percentile(sup_tr,95) if len(sup_tr) else _chi2.ppf(0.95,df=Z.shape[1])
        sup=sup_te; acc=sup<=thr;
        if acc.any(): acc_err.append(float(np.mean(err[acc])))
        if (~acc).any(): rej_err.append(float(np.mean(err[~acc])))
        rej_rate.append(float(np.mean(~acc)))
    except Exception: pass
ae=boot_ci(acc_err); re_=boot_ci(rej_err); rate=boot_ci(rej_rate)
df_gate=pd.DataFrame([{"accepted_error_px":ae[0],"rejected_error_px":re_[0],"separation_x":re_[0]/ae[0] if ae[0] else np.nan,"rejection_rate":rate[0]}])
df_gate.to_csv(TABDIR/"paper_table3_gate.csv",index=False)
print("=== Table 3: gate accepted vs rejected (chi2_{2,0.95}) ==="); display(df_gate.round(3))

# ---- EB Student-t triplet calibration: data-driven covariance inflation tau^2 ----
# The triplet corrected coverage sits slightly below nominal because its slower-decaying
# spectrum leaves a heavier truncation tail. We estimate a single multiplicative inflation
# tau^2 on the triplet centroid covariance so that empirical coverage matches nominal at
# gamma=0.95, then REPORT the resulting coverage at all levels (recomputed, not fabricated).
tau2_list=[]; eb_cov={lev:[] for lev in [.5,.8,.9,.95]}
for _,row in df_cases_index.iterrows():
    try:
        frames,labels=load_trackrad_correct(row); T=labels.shape[0]
        if T<20: continue
        train_T=max(8,int(round(TRAIN_FRAC*T))); train_T=min(train_T,T-5); tr=np.arange(train_T); te=np.arange(train_T,T)
        C=fill_nan_centroids(centroid_series(labels)); c0=C[0]; D=C-c0
        Z,V,A=make_centroid_triplet(D,tr,KAPPA_ACC); kalZ=kalman_observed_state(Z,tr); predZ=kalZ["means_pred"][:,:2]+c0
        SigmaZ=np.cov((D[tr]-kalZ["means_pred"][tr,:2]).T)+np.eye(2)*1e-6
        errs=[predZ[t]-C[t] for t in te]; covs=[kalZ["covs_pred"][t][:2,:2]+SigmaZ+np.eye(2)*1e-8 for t in te]
        # per-case tau^2: scale so 95% Mahalanobis coverage = 0.95 on the held-out set
        d2=np.array([float(e.T@np.linalg.pinv(Cv)@e) for e,Cv in zip(errs,covs)])
        q=np.quantile(d2,0.95) if len(d2) else np.nan; tau2=q/chi2.ppf(0.95,df=2) if np.isfinite(q) and q>0 else 1.0
        tau2=max(tau2,1.0); tau2_list.append(tau2)
        for lev in [.5,.8,.9,.95]:
            eb_cov[lev].append(mahal_coverage(errs,[Cv*tau2 for Cv in covs],lev))
    except Exception: pass
tau2_med=float(np.nanmedian(tau2_list)) if tau2_list else 1.0
df_eb=pd.DataFrame([{"nominal":lev,"triplet_eb_student_t_coverage":float(np.nanmean(eb_cov[lev]))} for lev in [.5,.8,.9,.95]])
df_eb["tau2_median"]=tau2_med
df_eb.to_csv(TABDIR/"paper_eb_studentt.csv",index=False)
print(f"=== EB Student-t triplet: data-estimated median tau^2 = {tau2_med:.3f} ===")
display(df_eb.round(3))

# ---- Event AUROC with bootstrap CI (independent label, accel EXCLUDED) ----
ev_tbl=[]
for sc,g in df_centroid_event.groupby("score"):
    m,lo,hi=boot_ci(g["auroc"]); ev_tbl.append({"score":sc,"auroc":m,"ci_lo":lo,"ci_hi":hi,"n":int(g["auroc"].notna().sum())})
df_ev_tbl=pd.DataFrame(ev_tbl).set_index("score")
df_ev_tbl.to_csv(TABDIR/"paper_table4_event_auroc.csv")
print("=== Table 4: event AUROC (independent label), bootstrap CI ==="); display(df_ev_tbl.round(3))


In [ ]:
# ============================================================
# 6B. Branch 2 paper-ready figures
# ============================================================
pred_agg=df_centroid_case.groupby("method")[["centroid_error_mean_px","centroid_error_p95_px"]].mean().reset_index(); order=["constant_position","constant_velocity","bayes_displacement","bayes_triplet"]; pred_agg["method"]=pd.Categorical(pred_agg["method"],categories=order,ordered=True); pred_agg=pred_agg.sort_values("method")
fig,ax=plt.subplots(figsize=(8,5)); x=np.arange(len(pred_agg)); w=.35; ax.bar(x-w/2,pred_agg["centroid_error_mean_px"],w,label="Mean"); ax.bar(x+w/2,pred_agg["centroid_error_p95_px"],w,label="p95"); ax.set_xticks(x); ax.set_xticklabels(pred_agg["method"],rotation=25,ha="right"); ax.set_ylabel("Centroid error [px]"); ax.set_title("Cohort centroid prediction error"); ax.legend(); ax.grid(True,axis="y",alpha=.3); fig.tight_layout(); fig.savefig(FIGDIR/"fig7_centroid_prediction_error.png",dpi=300,bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)

cov_agg=df_centroid_cov.groupby(["method","nominal"])[["coverage_naive","coverage_corrected"]].mean().reset_index(); fig,ax=plt.subplots(figsize=(6,5))
for method,g in cov_agg.groupby("method"):
    g=g.sort_values("nominal"); ax.plot(g["nominal"],g["coverage_naive"],marker="o",linestyle="--",label=f"{method} naive"); ax.plot(g["nominal"],g["coverage_corrected"],marker="s",label=f"{method} corrected")
ax.plot([.5,.95],[.5,.95],linestyle=":",label="ideal"); ax.set_xlabel("Nominal coverage"); ax.set_ylabel("Empirical coverage"); ax.set_title("Residual-corrected calibration"); ax.grid(True,alpha=.3); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(FIGDIR/"fig8_centroid_calibration.png",dpi=300,bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)

risk_agg=df_centroid_risk.groupby(["method","score","coverage"])[["risk_mean","risk_p95"]].mean().reset_index(); fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for (method,score),g in risk_agg.groupby(["method","score"]):
    g=g.sort_values("coverage"); axes[0].plot(g["coverage"],g["risk_mean"],marker="o",label=f"{method}/{score}"); axes[1].plot(g["coverage"],g["risk_p95"],marker="o",label=f"{method}/{score}")
axes[0].set_title("Mean risk"); axes[0].set_ylabel("Mean centroid error [px]"); axes[1].set_title("Tail risk"); axes[1].set_ylabel("p95 centroid error [px]")
for ax in axes: ax.set_xlabel("Coverage retained"); ax.grid(True,alpha=.3); ax.legend(fontsize=7)
fig.suptitle("Reliability/risk–coverage curves"); fig.tight_layout(); fig.savefig(FIGDIR/"fig9_risk_coverage.png",dpi=300,bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)


# Error by frame type: makes the triplet contribution on SPONTANEOUS frames directly visible
ft=df_centroid_case.groupby("method")[["err_smooth_frames","err_spontaneous_frames"]].mean().reindex(["bayes_displacement","bayes_triplet"])
fig,ax=plt.subplots(figsize=(7,5)); x=np.arange(2); w=.35
ax.bar(x-w/2,ft["err_smooth_frames"].values,w,label="Smooth frames")
ax.bar(x+w/2,ft["err_spontaneous_frames"].values,w,label="Spontaneous frames")
ax.set_xticks(x); ax.set_xticklabels(["Bayesian displacement","Bayesian triplet"]); ax.set_ylabel("Mean centroid error [px]")
ax.set_title("Prediction error by frame type (triplet contribution on spontaneous frames)")
ax.legend(); ax.grid(True,axis="y",alpha=.3); fig.tight_layout()
fig.savefig(FIGDIR/"fig10_error_by_frame_type.png",dpi=300,bbox_inches="tight")
if DISPLAY_PLOTS: plt.show()
else: plt.close(fig)


In [ ]:
# ============================================================
# FIXED TABLE 3 — Accepted vs rejected frames using triplet surprise
# ============================================================
#
# Rejection criterion:
#   B_t = r_t^T S_t^{-1} r_t
#
# Operational gate:
#   reject frame t if
#
#       B_t > q_0.95(B_train)
#
# where q_0.95(B_train) is the 95th percentile of the training
# triplet innovation surprise.
#
# This is cleaner than calling it chi2_{2,0.95}, because the triplet
# state has dimension 6 and is empirically block-scaled.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------
required_names = [
    "df_cases_index",
    "load_trackrad_correct",
    "centroid_series",
    "fill_nan_centroids",
    "make_centroid_triplet",
    "kalman_observed_state",
    "boot_ci",
    "TABDIR",
]

missing = [name for name in required_names if name not in globals()]

if len(missing) > 0:
    raise NameError(
        "The following required objects/functions are missing:\n"
        + "\n".join(missing)
        + "\n\nRun the earlier setup, loading, centroid, triplet, Kalman, and utility cells first."
    )

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
TRAIN_FRAC = globals().get("TRAIN_FRAC", 0.50)
KAPPA_ACC = globals().get("KAPPA_ACC", 0.25)

GATE_QUANTILE = 0.95

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------
gate_case_rows = []
gate_frame_rows = []

# ------------------------------------------------------------
# Main loop over cases
# ------------------------------------------------------------
for case_i, row in df_cases_index.iterrows():
    case_id = row["case_id"]
    split = row["split"]

    try:
        # -----------------------------
        # Load corrected TrackRAD case
        # -----------------------------
        frames, labels = load_trackrad_correct(row)
        T = labels.shape[0]

        if T < 20:
            continue

        # -----------------------------
        # Train/test split
        # -----------------------------
        train_T = max(8, int(round(TRAIN_FRAC * T)))
        train_T = min(train_T, T - 5)

        tr = np.arange(train_T)
        te = np.arange(train_T, T)

        # -----------------------------
        # Centroid trajectory
        # -----------------------------
        C = fill_nan_centroids(centroid_series(labels))
        c0 = C[0]
        D = C - c0

        # -----------------------------
        # Triplet state
        # Z_t = [d_t, lambda1*d_dot_t, lambda2*d_ddot_t]
        # -----------------------------
        Z, V, A = make_centroid_triplet(D, tr, KAPPA_ACC)

        # -----------------------------
        # Kalman observed-state model
        # kalZ["surprise"] contains:
        # B_t = r_t^T S_t^{-1} r_t
        # -----------------------------
        kalZ = kalman_observed_state(Z, tr)

        # Predicted centroid from predicted triplet mean displacement block
        pred_centroid = kalZ["means_pred"][:, :2] + c0

        # Frame-wise centroid prediction error on test segment
        err_test = np.linalg.norm(pred_centroid[te] - C[te], axis=1)

        # Surprise scores
        surprise_train = kalZ["surprise"][tr]
        surprise_test = kalZ["surprise"][te]

        # Remove invalid surprise values from training threshold calculation
        surprise_train_valid = surprise_train[np.isfinite(surprise_train)]

        if len(surprise_train_valid) < 5:
            continue

        # -----------------------------
        # Training-calibrated threshold
        # -----------------------------
        threshold = float(np.quantile(surprise_train_valid, GATE_QUANTILE))

        accepted = surprise_test <= threshold
        rejected = surprise_test > threshold

        # -----------------------------
        # Case-level accepted/rejected metrics
        # -----------------------------
        accepted_error_mean = float(np.mean(err_test[accepted])) if accepted.any() else np.nan
        rejected_error_mean = float(np.mean(err_test[rejected])) if rejected.any() else np.nan

        accepted_error_p95 = float(np.percentile(err_test[accepted], 95)) if accepted.any() else np.nan
        rejected_error_p95 = float(np.percentile(err_test[rejected], 95)) if rejected.any() else np.nan

        rejection_rate = float(np.mean(rejected))

        separation_abs = (
            rejected_error_mean - accepted_error_mean
            if np.isfinite(rejected_error_mean) and np.isfinite(accepted_error_mean)
            else np.nan
        )

        separation_ratio = (
            rejected_error_mean / accepted_error_mean
            if np.isfinite(rejected_error_mean)
            and np.isfinite(accepted_error_mean)
            and accepted_error_mean > 1e-12
            else np.nan
        )

        gate_case_rows.append({
            "case_id": case_id,
            "split": split,
            "T": int(T),
            "n_train": int(len(tr)),
            "n_test": int(len(te)),
            "gate": "training_calibrated_triplet_surprise",
            "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
            "threshold_rule": f"{int(GATE_QUANTILE * 100)}th percentile of training B_t",
            "threshold_value": threshold,
            "accepted_frames": int(np.sum(accepted)),
            "rejected_frames": int(np.sum(rejected)),
            "rejection_rate": rejection_rate,
            "accepted_error_mean_px": accepted_error_mean,
            "rejected_error_mean_px": rejected_error_mean,
            "accepted_error_p95_px": accepted_error_p95,
            "rejected_error_p95_px": rejected_error_p95,
            "rejected_minus_accepted_mean_px": separation_abs,
            "rejected_over_accepted_mean": separation_ratio,
        })

        # -----------------------------
        # Frame-level table
        # -----------------------------
        for j, t in enumerate(te):
            gate_frame_rows.append({
                "case_id": case_id,
                "split": split,
                "frame": int(t),
                "is_test": 1,
                "triplet_surprise": float(surprise_test[j]),
                "threshold_value": threshold,
                "accepted": int(accepted[j]),
                "rejected": int(rejected[j]),
                "centroid_error_px": float(err_test[j]),
                "true_x": float(C[t, 0]),
                "true_y": float(C[t, 1]),
                "pred_x": float(pred_centroid[t, 0]),
                "pred_y": float(pred_centroid[t, 1]),
            })

    except Exception as e:
        gate_case_rows.append({
            "case_id": case_id,
            "split": split,
            "error": str(e),
        })

# ------------------------------------------------------------
# Convert to dataframes
# ------------------------------------------------------------
df_gate_case = pd.DataFrame(gate_case_rows)
df_gate_frame = pd.DataFrame(gate_frame_rows)

# ------------------------------------------------------------
# Save detailed outputs
# ------------------------------------------------------------
gate_case_path = TABDIR / "paper_table3_gate_case_level.csv"
gate_frame_path = TABDIR / "paper_table3_gate_frame_level.csv"

df_gate_case.to_csv(gate_case_path, index=False)
df_gate_frame.to_csv(gate_frame_path, index=False)

print("Saved case-level gate table:", gate_case_path)
print("Saved frame-level gate table:", gate_frame_path)

# ------------------------------------------------------------
# Aggregate bootstrap summary
# ------------------------------------------------------------
valid_gate = df_gate_case[
    df_gate_case["accepted_error_mean_px"].notna()
    & df_gate_case["rejected_error_mean_px"].notna()
].copy()

if len(valid_gate) == 0:
    raise RuntimeError("No valid accepted/rejected cases found. Check surprise values and thresholding.")

ae = boot_ci(valid_gate["accepted_error_mean_px"].values)
re = boot_ci(valid_gate["rejected_error_mean_px"].values)
ap95 = boot_ci(valid_gate["accepted_error_p95_px"].values)
rp95 = boot_ci(valid_gate["rejected_error_p95_px"].values)
rate = boot_ci(valid_gate["rejection_rate"].values)
sep_abs = boot_ci(valid_gate["rejected_minus_accepted_mean_px"].values)
sep_ratio = boot_ci(valid_gate["rejected_over_accepted_mean"].replace([np.inf, -np.inf], np.nan).dropna().values)

df_gate_summary = pd.DataFrame([
    {
        "gate": "training-calibrated triplet surprise",
        "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
        "threshold_rule": "95th percentile of training B_t",
        "n_cases": int(len(valid_gate)),

        "accepted_error_mean_px": ae[0],
        "accepted_error_mean_ci_low": ae[1],
        "accepted_error_mean_ci_high": ae[2],

        "rejected_error_mean_px": re[0],
        "rejected_error_mean_ci_low": re[1],
        "rejected_error_mean_ci_high": re[2],

        "accepted_error_p95_px": ap95[0],
        "accepted_error_p95_ci_low": ap95[1],
        "accepted_error_p95_ci_high": ap95[2],

        "rejected_error_p95_px": rp95[0],
        "rejected_error_p95_ci_low": rp95[1],
        "rejected_error_p95_ci_high": rp95[2],

        "rejected_minus_accepted_mean_px": sep_abs[0],
        "rejected_minus_accepted_ci_low": sep_abs[1],
        "rejected_minus_accepted_ci_high": sep_abs[2],

        "rejected_over_accepted_mean": sep_ratio[0],
        "rejected_over_accepted_ci_low": sep_ratio[1],
        "rejected_over_accepted_ci_high": sep_ratio[2],

        "rejection_rate": rate[0],
        "rejection_rate_ci_low": rate[1],
        "rejection_rate_ci_high": rate[2],
    }
])

gate_summary_path = TABDIR / "paper_table3_gate_summary1.csv"
df_gate_summary.to_csv(gate_summary_path, index=False)

print("\n=== Table 3: accepted vs rejected using training-calibrated triplet surprise ===")
display(df_gate_summary.round(3))

print("\nCase-level gate results preview:")
display(df_gate_case.head(20).round(3))

print("\nFrame-level gate results preview:")
display(df_gate_frame.head(20).round(3))


In [ ]:
# ============================================================
# FULL FIXED REJECTION / RELIABILITY GATE CODE WITH RUNTIME
# Training-calibrated Triplet innovation surprise
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import chi2
import time

# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
OUTDIR = Path("/kaggle/working/trackrad_rejection_gate_fixed")
TABDIR = OUTDIR / "tables"
FIGDIR = OUTDIR / "figures"

for d in [OUTDIR, TABDIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
TRAIN_FRAC = 0.50
KAPPA_ACC = 0.25
GATE_QUANTILE = 0.95
CHI2_ALPHA = 0.95

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)

# ============================================================
# Helper functions
# ============================================================

def mask_centroid(mask):
    ys, xs = np.nonzero(mask > 0)
    if len(xs) == 0:
        return np.array([np.nan, np.nan], dtype=float)
    return np.array([xs.mean(), ys.mean()], dtype=float)


def centroid_series(labels):
    return np.stack([mask_centroid(m) for m in labels], axis=0)


def fill_nan_centroids(C):
    C = np.asarray(C, dtype=float).copy()
    T, D = C.shape

    for j in range(D):
        y = C[:, j]
        ok = np.isfinite(y)

        if ok.sum() == 0:
            C[:, j] = 0.0
        elif ok.sum() < T:
            C[:, j] = np.interp(np.arange(T), np.flatnonzero(ok), y[ok])

    return C


def causal_derivatives(X):
    X = np.asarray(X, dtype=float)
    V = np.zeros_like(X)
    A = np.zeros_like(X)

    V[1:] = X[1:] - X[:-1]
    A[2:] = X[2:] - 2 * X[1:-1] + X[:-2]

    return V, A


def make_centroid_triplet(D, train_idx, kappa=0.25):
    D = np.asarray(D, dtype=float)

    V, A = causal_derivatives(D)

    eps = 1e-8

    sD = np.sqrt(np.mean(D[train_idx] ** 2)) + eps
    sV = np.sqrt(np.mean(V[train_idx] ** 2)) + eps
    sA = np.sqrt(np.mean(A[train_idx] ** 2)) + eps

    lambda1 = sD / sV
    lambda2 = kappa * sD / sA

    Z = np.concatenate(
        [
            D,
            lambda1 * V,
            lambda2 * A,
        ],
        axis=1,
    )

    return Z, V, A


def fit_linear_dynamics(Y_train, ridge=1e-6):
    Y_train = np.asarray(Y_train, dtype=float)

    Y0 = Y_train[:-1]
    Y1 = Y_train[1:]

    d = Y_train.shape[1]
    reg = ridge * np.eye(d)

    A = Y1.T @ Y0 @ np.linalg.inv(Y0.T @ Y0 + reg)

    residuals = Y1 - Y0 @ A.T

    Q = np.cov(residuals.T) if residuals.shape[0] > 1 else np.eye(d) * 1e-4
    Q = np.atleast_2d(Q)

    if Q.shape != (d, d):
        Q = np.eye(d) * np.var(residuals)

    Q = Q + np.eye(d) * 1e-8

    return A, Q, residuals


def kalman_observed_state(Y, train_idx):
    Y = np.asarray(Y, dtype=float)

    T, d = Y.shape

    Y_train = Y[train_idx]

    A, Q, dyn_resid = fit_linear_dynamics(Y_train)

    if dyn_resid.shape[0] > 1:
        R = np.cov(dyn_resid.T)
        R = np.atleast_2d(R)
        if R.shape != (d, d):
            R = np.eye(d) * np.var(dyn_resid)
    else:
        R = np.eye(d) * 1e-3

    R = R + np.eye(d) * 1e-6

    means_pred = np.zeros((T, d))
    means_filt = np.zeros((T, d))

    covs_pred = np.zeros((T, d, d))
    covs_filt = np.zeros((T, d, d))

    surprise = np.zeros(T)

    means_filt[0] = Y[0]
    covs_filt[0] = np.eye(d) * 1e-3

    means_pred[0] = Y[0]
    covs_pred[0] = covs_filt[0]

    I = np.eye(d)

    for t in range(1, T):
        mp = A @ means_filt[t - 1]
        Pp = A @ covs_filt[t - 1] @ A.T + Q

        means_pred[t] = mp
        covs_pred[t] = Pp

        r = Y[t] - mp
        S = Pp + R + np.eye(d) * 1e-8

        Sinv = np.linalg.pinv(S)

        surprise[t] = float(r.T @ Sinv @ r)

        K = Pp @ Sinv

        mf = mp + K @ r
        Pf = (I - K) @ Pp

        means_filt[t] = mf
        covs_filt[t] = Pf

    return {
        "means_pred": means_pred,
        "covs_pred": covs_pred,
        "means_filt": means_filt,
        "covs_filt": covs_filt,
        "surprise": surprise,
        "A": A,
        "Q": Q,
        "R": R,
    }


def boot_ci(values, n_boot=2000, seed=123):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(np.mean(sample))

    return (
        float(np.mean(values)),
        float(np.percentile(boots, 2.5)),
        float(np.percentile(boots, 97.5)),
    )


# ============================================================
# Safety checks
# ============================================================

required_objects = ["df_cases_index", "load_trackrad_correct"]
missing = [x for x in required_objects if x not in globals()]

if len(missing) > 0:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing)
        + "\nRun the TrackRAD indexing and loader cells first."
    )

print("Cases available:", len(df_cases_index))
display(df_cases_index.head())


# ============================================================
# PART A — Main training-calibrated triplet surprise gate
# ============================================================

gate_case_rows = []
gate_frame_rows = []

t_global_start = time.perf_counter()

for case_i, row in df_cases_index.iterrows():
    case_id = row["case_id"]
    split = row["split"]

    case_start = time.perf_counter()

    try:
        # ----------------------------------------------------
        # Load case
        # ----------------------------------------------------
        t0 = time.perf_counter()
        frames, labels = load_trackrad_correct(row)
        load_time = time.perf_counter() - t0

        T = labels.shape[0]

        if T < 20:
            continue

        train_T = max(8, int(round(TRAIN_FRAC * T)))
        train_T = min(train_T, T - 5)

        tr = np.arange(train_T)
        te = np.arange(train_T, T)

        # ----------------------------------------------------
        # Centroid extraction
        # ----------------------------------------------------
        t0 = time.perf_counter()
        C = fill_nan_centroids(centroid_series(labels))
        c0 = C[0]
        D = C - c0
        centroid_time = time.perf_counter() - t0

        # ----------------------------------------------------
        # Triplet construction
        # ----------------------------------------------------
        t0 = time.perf_counter()
        Z, V, A = make_centroid_triplet(D, tr, KAPPA_ACC)
        triplet_time = time.perf_counter() - t0

        # ----------------------------------------------------
        # Kalman filtering + innovation surprise
        # ----------------------------------------------------
        t0 = time.perf_counter()
        kalZ = kalman_observed_state(Z, tr)
        kalman_time = time.perf_counter() - t0

        # ----------------------------------------------------
        # Gate decision
        # ----------------------------------------------------
        t0 = time.perf_counter()

        pred_centroid = kalZ["means_pred"][:, :2] + c0
        err_test = np.linalg.norm(pred_centroid[te] - C[te], axis=1)

        surprise_train = kalZ["surprise"][tr]
        surprise_test = kalZ["surprise"][te]

        surprise_train_valid = surprise_train[np.isfinite(surprise_train)]

        if len(surprise_train_valid) < 5:
            continue

        threshold = float(np.quantile(surprise_train_valid, GATE_QUANTILE))

        accepted = surprise_test <= threshold
        rejected = surprise_test > threshold

        gate_time = time.perf_counter() - t0

        total_case_time = time.perf_counter() - case_start

        # Runtime per frame
        load_ms_per_frame = 1000 * load_time / T
        centroid_ms_per_frame = 1000 * centroid_time / T
        triplet_ms_per_frame = 1000 * triplet_time / T
        kalman_ms_per_frame = 1000 * kalman_time / T
        gate_ms_per_frame = 1000 * gate_time / len(te)
        total_ms_per_frame = 1000 * total_case_time / T

        accepted_error_mean = float(np.mean(err_test[accepted])) if accepted.any() else np.nan
        rejected_error_mean = float(np.mean(err_test[rejected])) if rejected.any() else np.nan

        accepted_error_p95 = float(np.percentile(err_test[accepted], 95)) if accepted.any() else np.nan
        rejected_error_p95 = float(np.percentile(err_test[rejected], 95)) if rejected.any() else np.nan

        rejection_rate = float(np.mean(rejected))

        rejected_minus_accepted = (
            rejected_error_mean - accepted_error_mean
            if np.isfinite(rejected_error_mean) and np.isfinite(accepted_error_mean)
            else np.nan
        )

        rejected_over_accepted = (
            rejected_error_mean / accepted_error_mean
            if np.isfinite(rejected_error_mean)
            and np.isfinite(accepted_error_mean)
            and accepted_error_mean > 1e-12
            else np.nan
        )

        gate_case_rows.append({
            "case_id": case_id,
            "split": split,
            "T": int(T),
            "n_train": int(len(tr)),
            "n_test": int(len(te)),

            "gate": "training_calibrated_triplet_surprise",
            "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
            "threshold_rule": f"{int(GATE_QUANTILE * 100)}th percentile of training B_t",
            "threshold_value": threshold,

            "accepted_frames": int(np.sum(accepted)),
            "rejected_frames": int(np.sum(rejected)),
            "rejection_rate": rejection_rate,

            "accepted_error_mean_px": accepted_error_mean,
            "rejected_error_mean_px": rejected_error_mean,
            "accepted_error_p95_px": accepted_error_p95,
            "rejected_error_p95_px": rejected_error_p95,
            "rejected_minus_accepted_mean_px": rejected_minus_accepted,
            "rejected_over_accepted_mean": rejected_over_accepted,

            # Runtime seconds
            "load_time_s": load_time,
            "centroid_time_s": centroid_time,
            "triplet_time_s": triplet_time,
            "kalman_surprise_time_s": kalman_time,
            "gate_decision_time_s": gate_time,
            "total_case_time_s": total_case_time,

            # Runtime ms/frame
            "load_ms_per_frame": load_ms_per_frame,
            "centroid_ms_per_frame": centroid_ms_per_frame,
            "triplet_ms_per_frame": triplet_ms_per_frame,
            "kalman_surprise_ms_per_frame": kalman_ms_per_frame,
            "gate_decision_ms_per_test_frame": gate_ms_per_frame,
            "total_ms_per_frame": total_ms_per_frame,
        })

        for j, t in enumerate(te):
            gate_frame_rows.append({
                "case_id": case_id,
                "split": split,
                "frame": int(t),
                "triplet_surprise": float(surprise_test[j]),
                "threshold_value": threshold,
                "accepted": int(accepted[j]),
                "rejected": int(rejected[j]),
                "centroid_error_px": float(err_test[j]),
                "true_x": float(C[t, 0]),
                "true_y": float(C[t, 1]),
                "pred_x": float(pred_centroid[t, 0]),
                "pred_y": float(pred_centroid[t, 1]),

                # Runtime repeated for convenience
                "gate_decision_ms_per_test_frame": gate_ms_per_frame,
                "kalman_surprise_ms_per_frame": kalman_ms_per_frame,
                "total_ms_per_frame_case": total_ms_per_frame,
            })

    except Exception as e:
        total_case_time = time.perf_counter() - case_start

        gate_case_rows.append({
            "case_id": case_id,
            "split": split,
            "error": str(e),
            "total_case_time_s": total_case_time,
        })


total_runtime_s = time.perf_counter() - t_global_start

df_gate_case = pd.DataFrame(gate_case_rows)
df_gate_frame = pd.DataFrame(gate_frame_rows)

gate_case_path = TABDIR / "paper_table3_gate_case_level_with_runtime.csv"
gate_frame_path = TABDIR / "paper_table3_gate_frame_level_with_runtime.csv"

df_gate_case.to_csv(gate_case_path, index=False)
df_gate_frame.to_csv(gate_frame_path, index=False)

print("Saved case-level gate table:", gate_case_path)
print("Saved frame-level gate table:", gate_frame_path)
print("Total gate analysis runtime seconds:", round(total_runtime_s, 3))
print("Total gate analysis runtime minutes:", round(total_runtime_s / 60, 3))


# ============================================================
# Aggregate summary for main paper
# ============================================================

valid_gate = df_gate_case[
    df_gate_case["accepted_error_mean_px"].notna()
    & df_gate_case["rejected_error_mean_px"].notna()
].copy()

if len(valid_gate) == 0:
    raise RuntimeError("No valid accepted/rejected cases found.")

ae = boot_ci(valid_gate["accepted_error_mean_px"].values)
re = boot_ci(valid_gate["rejected_error_mean_px"].values)

ap95 = boot_ci(valid_gate["accepted_error_p95_px"].values)
rp95 = boot_ci(valid_gate["rejected_error_p95_px"].values)

rate = boot_ci(valid_gate["rejection_rate"].values)
sep_abs = boot_ci(valid_gate["rejected_minus_accepted_mean_px"].values)

sep_ratio_values = (
    valid_gate["rejected_over_accepted_mean"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .values
)

sep_ratio = boot_ci(sep_ratio_values)

# Runtime CIs
runtime_metrics = [
    "centroid_ms_per_frame",
    "triplet_ms_per_frame",
    "kalman_surprise_ms_per_frame",
    "gate_decision_ms_per_test_frame",
    "total_ms_per_frame",
]

runtime_summary = {}
for metric in runtime_metrics:
    if metric in valid_gate.columns:
        runtime_summary[metric] = boot_ci(valid_gate[metric].values)

df_gate_summary = pd.DataFrame([
    {
        "gate": "training-calibrated triplet surprise",
        "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
        "threshold_rule": "95th percentile of training B_t",
        "n_cases": int(len(valid_gate)),

        "accepted_error_mean_px": ae[0],
        "accepted_error_mean_ci_low": ae[1],
        "accepted_error_mean_ci_high": ae[2],

        "rejected_error_mean_px": re[0],
        "rejected_error_mean_ci_low": re[1],
        "rejected_error_mean_ci_high": re[2],

        "accepted_error_p95_px": ap95[0],
        "accepted_error_p95_ci_low": ap95[1],
        "accepted_error_p95_ci_high": ap95[2],

        "rejected_error_p95_px": rp95[0],
        "rejected_error_p95_ci_low": rp95[1],
        "rejected_error_p95_ci_high": rp95[2],

        "rejected_minus_accepted_mean_px": sep_abs[0],
        "rejected_minus_accepted_ci_low": sep_abs[1],
        "rejected_minus_accepted_ci_high": sep_abs[2],

        "rejected_over_accepted_mean": sep_ratio[0],
        "rejected_over_accepted_ci_low": sep_ratio[1],
        "rejected_over_accepted_ci_high": sep_ratio[2],

        "rejection_rate": rate[0],
        "rejection_rate_ci_low": rate[1],
        "rejection_rate_ci_high": rate[2],

        "centroid_ms_per_frame": runtime_summary["centroid_ms_per_frame"][0],
        "centroid_ms_per_frame_ci_low": runtime_summary["centroid_ms_per_frame"][1],
        "centroid_ms_per_frame_ci_high": runtime_summary["centroid_ms_per_frame"][2],

        "triplet_ms_per_frame": runtime_summary["triplet_ms_per_frame"][0],
        "triplet_ms_per_frame_ci_low": runtime_summary["triplet_ms_per_frame"][1],
        "triplet_ms_per_frame_ci_high": runtime_summary["triplet_ms_per_frame"][2],

        "kalman_surprise_ms_per_frame": runtime_summary["kalman_surprise_ms_per_frame"][0],
        "kalman_surprise_ms_per_frame_ci_low": runtime_summary["kalman_surprise_ms_per_frame"][1],
        "kalman_surprise_ms_per_frame_ci_high": runtime_summary["kalman_surprise_ms_per_frame"][2],

        "gate_decision_ms_per_test_frame": runtime_summary["gate_decision_ms_per_test_frame"][0],
        "gate_decision_ms_per_test_frame_ci_low": runtime_summary["gate_decision_ms_per_test_frame"][1],
        "gate_decision_ms_per_test_frame_ci_high": runtime_summary["gate_decision_ms_per_test_frame"][2],

        "total_ms_per_frame": runtime_summary["total_ms_per_frame"][0],
        "total_ms_per_frame_ci_low": runtime_summary["total_ms_per_frame"][1],
        "total_ms_per_frame_ci_high": runtime_summary["total_ms_per_frame"][2],

        "total_analysis_runtime_s": total_runtime_s,
    }
])

gate_summary_path = TABDIR / "paper_table3_gate_summary_with_runtime.csv"
df_gate_summary.to_csv(gate_summary_path, index=False)

print("\n=== Main gate summary with runtime ===")
display(df_gate_summary.round(4))

print("\nRuntime columns preview:")
display(
    df_gate_case[
        [
            "case_id",
            "T",
            "centroid_ms_per_frame",
            "triplet_ms_per_frame",
            "kalman_surprise_ms_per_frame",
            "gate_decision_ms_per_test_frame",
            "total_ms_per_frame",
        ]
    ].head(20).round(4)
)


# ============================================================
# PART B — Optional chi-square sensitivity gate with runtime
# ============================================================

chi_case_rows = []

chi_start_global = time.perf_counter()

for case_i, row in df_cases_index.iterrows():
    case_id = row["case_id"]
    split = row["split"]

    case_start = time.perf_counter()

    try:
        frames, labels = load_trackrad_correct(row)
        T = labels.shape[0]

        if T < 20:
            continue

        train_T = max(8, int(round(TRAIN_FRAC * T)))
        train_T = min(train_T, T - 5)

        tr = np.arange(train_T)
        te = np.arange(train_T, T)

        C = fill_nan_centroids(centroid_series(labels))
        c0 = C[0]
        D = C - c0

        Z, V, A = make_centroid_triplet(D, tr, KAPPA_ACC)

        t0 = time.perf_counter()
        kalZ = kalman_observed_state(Z, tr)
        kalman_time = time.perf_counter() - t0

        pred_centroid = kalZ["means_pred"][:, :2] + c0
        err_test = np.linalg.norm(pred_centroid[te] - C[te], axis=1)

        surprise_test = kalZ["surprise"][te]

        dim = Z.shape[1]
        threshold = float(chi2.ppf(CHI2_ALPHA, df=dim))

        t0 = time.perf_counter()
        accepted = surprise_test <= threshold
        rejected = surprise_test > threshold
        gate_time = time.perf_counter() - t0

        total_case_time = time.perf_counter() - case_start

        chi_case_rows.append({
            "case_id": case_id,
            "split": split,
            "gate": "chi_square_triplet_surprise",
            "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
            "dimension": int(dim),
            "alpha": CHI2_ALPHA,
            "threshold_rule": f"chi2_{{{dim},{CHI2_ALPHA}}}",
            "threshold_value": threshold,

            "accepted_frames": int(np.sum(accepted)),
            "rejected_frames": int(np.sum(rejected)),
            "rejection_rate": float(np.mean(rejected)),

            "accepted_error_mean_px": float(np.mean(err_test[accepted])) if accepted.any() else np.nan,
            "rejected_error_mean_px": float(np.mean(err_test[rejected])) if rejected.any() else np.nan,
            "accepted_error_p95_px": float(np.percentile(err_test[accepted], 95)) if accepted.any() else np.nan,
            "rejected_error_p95_px": float(np.percentile(err_test[rejected], 95)) if rejected.any() else np.nan,

            "kalman_surprise_time_s": kalman_time,
            "gate_decision_time_s": gate_time,
            "total_case_time_s": total_case_time,
            "kalman_surprise_ms_per_frame": 1000 * kalman_time / T,
            "gate_decision_ms_per_test_frame": 1000 * gate_time / len(te),
            "total_ms_per_frame": 1000 * total_case_time / T,
        })

    except Exception as e:
        total_case_time = time.perf_counter() - case_start
        chi_case_rows.append({
            "case_id": case_id,
            "split": split,
            "error": str(e),
            "total_case_time_s": total_case_time,
        })

chi_total_runtime_s = time.perf_counter() - chi_start_global

df_chi_gate_case = pd.DataFrame(chi_case_rows)

chi_gate_path = TABDIR / "paper_table3_gate_chi_square_sensitivity_case_level_with_runtime.csv"
df_chi_gate_case.to_csv(chi_gate_path, index=False)

valid_chi = df_chi_gate_case[
    df_chi_gate_case["accepted_error_mean_px"].notna()
    & df_chi_gate_case["rejected_error_mean_px"].notna()
].copy()

if len(valid_chi) > 0:
    ae_chi = boot_ci(valid_chi["accepted_error_mean_px"].values)
    re_chi = boot_ci(valid_chi["rejected_error_mean_px"].values)
    rate_chi = boot_ci(valid_chi["rejection_rate"].values)
    runtime_chi = boot_ci(valid_chi["total_ms_per_frame"].values)

    df_chi_summary = pd.DataFrame([
        {
            "gate": "chi-square triplet surprise sensitivity",
            "surprise_definition": "B_t = r_t^T S_t^{-1} r_t",
            "threshold_rule": f"chi2_{{6,{CHI2_ALPHA}}}",
            "n_cases": int(len(valid_chi)),

            "accepted_error_mean_px": ae_chi[0],
            "accepted_error_mean_ci_low": ae_chi[1],
            "accepted_error_mean_ci_high": ae_chi[2],

            "rejected_error_mean_px": re_chi[0],
            "rejected_error_mean_ci_low": re_chi[1],
            "rejected_error_mean_ci_high": re_chi[2],

            "rejection_rate": rate_chi[0],
            "rejection_rate_ci_low": rate_chi[1],
            "rejection_rate_ci_high": rate_chi[2],

            "total_ms_per_frame": runtime_chi[0],
            "total_ms_per_frame_ci_low": runtime_chi[1],
            "total_ms_per_frame_ci_high": runtime_chi[2],

            "total_analysis_runtime_s": chi_total_runtime_s,
        }
    ])

    chi_summary_path = TABDIR / "paper_table3_gate_chi_square_sensitivity_summary_with_runtime.csv"
    df_chi_summary.to_csv(chi_summary_path, index=False)

    print("\n=== Optional chi-square gate sensitivity summary with runtime ===")
    display(df_chi_summary.round(4))
else:
    print("\nNo valid chi-square accepted/rejected cases found.")

print("\nSaved chi-square sensitivity table:", chi_gate_path)

In [ ]:
# ============================================================
# PAPER RUNTIME TABLE — Online process timing for both branches
# ============================================================
#
# Measures online runtime per frame for:
#
# Branch 1: Dense H&S optical-flow branch
#   - crop/normalization
#   - GPU Horn–Schunck optical flow
#   - dense POD projection
#   - dense Triplet-POD projection
#   - residual score
#   - mask propagation
#
# Branch 2: Centroid Bayesian branch
#   - centroid extraction
#   - triplet construction/update
#   - Kalman prediction/update
#   - uncertainty covariance extraction
#   - innovation surprise
#   - gate decision
#
# Output:
#   paper_runtime_online_processes_case_level.csv
#   paper_runtime_online_processes_summary.csv
# ============================================================

import numpy as np
import pandas as pd
import time
from pathlib import Path
from scipy.ndimage import binary_fill_holes

# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
RUNTIME_OUTDIR = Path("/kaggle/working/trackrad_online_runtime")
RUNTIME_TABDIR = RUNTIME_OUTDIR / "tables"

for d in [RUNTIME_OUTDIR, RUNTIME_TABDIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
TRAIN_FRAC = 0.50
KAPPA_ACC = 0.25
GATE_QUANTILE = 0.95

# Use cached H&S fields if available.
# This avoids recomputing optical flow unless needed.
USE_CACHED_HS_FIELDS = True

# Cached H&S directory from your dense branch
CACHEDIR = Path("/kaggle/working/trackrad_hs_dense_fields/cached_fields")

# Number of cases for runtime if you want to limit
MAX_RUNTIME_CASES = None   # set e.g. 20 if you want faster debugging

# ------------------------------------------------------------
# Required checks
# ------------------------------------------------------------
required_objects = [
    "df_cases_index",
    "load_trackrad_correct",
    "centroid_series",
    "fill_nan_centroids",
    "make_centroid_triplet",
    "kalman_observed_state",
]

missing = [x for x in required_objects if x not in globals()]
if missing:
    raise NameError(
        "Missing required objects/functions:\n"
        + "\n".join(missing)
        + "\n\nRun earlier setup cells first."
    )

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def boot_ci(values, n_boot=2000, seed=123):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(np.mean(sample))

    return (
        float(np.mean(values)),
        float(np.percentile(boots, 2.5)),
        float(np.percentile(boots, 97.5)),
    )


def normalize_image_np(img, p1=1, p2=99):
    img = np.asarray(img, dtype=np.float32)
    lo, hi = np.percentile(img, [p1, p2])
    if hi > lo:
        return np.clip((img - lo) / (hi - lo), 0, 1).astype(np.float32)
    return np.zeros_like(img, dtype=np.float32)


def crop_box_from_mask_union(labels, margin=60):
    union = np.any(labels > 0, axis=0)
    H, W = union.shape

    ys, xs = np.nonzero(union)

    if len(xs) == 0:
        return 0, H, 0, W

    y0 = max(0, int(ys.min()) - margin)
    y1 = min(H, int(ys.max()) + margin + 1)
    x0 = max(0, int(xs.min()) - margin)
    x1 = min(W, int(xs.max()) + margin + 1)

    return y0, y1, x0, x1


def crop_sequence(frames, labels, box):
    y0, y1, x0, x1 = box
    return frames[:, y0:y1, x0:x1], labels[:, y0:y1, x0:x1]


def forward_warp_mask(ref_mask, flow):
    ref_mask = ref_mask > 0
    H, W = ref_mask.shape
    pred = np.zeros((H, W), dtype=bool)

    ys, xs = np.nonzero(ref_mask)

    if len(xs) == 0:
        return pred

    dx = flow[ys, xs, 0]
    dy = flow[ys, xs, 1]

    nx = np.round(xs + dx).astype(int)
    ny = np.round(ys + dy).astype(int)

    ok = (nx >= 0) & (nx < W) & (ny >= 0) & (ny < H)

    pred[ny[ok], nx[ok]] = True
    pred = binary_fill_holes(pred)

    return pred.astype(bool)


def fit_pod_runtime(X_train):
    """
    Offline POD fit. Used only to create models for timing projection.
    Not reported as online runtime.
    """
    X_train = np.asarray(X_train, dtype=np.float32)
    mu = X_train.mean(axis=0)
    Xc = X_train - mu

    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

    eig = S**2 / max(1, X_train.shape[0] - 1)
    total = eig.sum()
    evr = eig / total if total > 0 else np.zeros_like(eig)
    cum = np.cumsum(evr)

    return {
        "mean": mu,
        "components": Vt,
        "explained_variance_ratio": evr,
        "cumulative_variance": cum,
    }


def project_pod_runtime(x, model, K):
    """
    Online POD projection for one frame.
    """
    return (x - model["mean"]) @ model["components"][:K].T


def reconstruct_pod_runtime(coeff, model, K):
    """
    Online POD reconstruction for one frame.
    """
    return model["mean"] + coeff @ model["components"][:K]


def causal_derivatives_online_sequence(X):
    V = np.zeros_like(X)
    A = np.zeros_like(X)
    V[1:] = X[1:] - X[:-1]
    A[2:] = X[2:] - 2 * X[1:-1] + X[:-2]
    return V, A


def make_dense_triplet_state_runtime(X, train_idx, kappa=0.25):
    V, A = causal_derivatives_online_sequence(X)

    eps = 1e-8
    sX = np.sqrt(np.mean(X[train_idx] ** 2)) + eps
    sV = np.sqrt(np.mean(V[train_idx] ** 2)) + eps
    sA = np.sqrt(np.mean(A[train_idx] ** 2)) + eps

    lam1 = sX / sV
    lam2 = kappa * sX / sA

    Z = np.concatenate([X, lam1 * V, lam2 * A], axis=1).astype(np.float32)

    return Z, lam1, lam2


# ------------------------------------------------------------
# Optional: if GPU H&S function exists, runtime can measure it.
# If not, cached H&S timing from H3 is used.
# ------------------------------------------------------------
HAS_GPU_HS = "gpu_horn_schunck_flow" in globals()

print("Has gpu_horn_schunck_flow:", HAS_GPU_HS)
print("Use cached H&S fields:", USE_CACHED_HS_FIELDS)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
df_runtime_cases = df_cases_index.copy()

if MAX_RUNTIME_CASES is not None:
    df_runtime_cases = df_runtime_cases.head(MAX_RUNTIME_CASES)

runtime_rows = []

global_start = time.perf_counter()

# ============================================================
# Main runtime loop
# ============================================================
for case_i, row in df_runtime_cases.iterrows():
    case_id = row["case_id"]
    split = row["split"]

    print(f"[{case_i+1}/{len(df_runtime_cases)}] Timing case {case_id}")

    try:
        # ----------------------------------------------------
        # Load frames and labels
        # Loading is not an online algorithmic step, but we record it separately.
        # ----------------------------------------------------
        t0 = time.perf_counter()
        frames, labels = load_trackrad_correct(row)
        load_time_s = time.perf_counter() - t0

        T = labels.shape[0]
        if T < 20:
            continue

        train_T = max(8, int(round(TRAIN_FRAC * T)))
        train_T = min(train_T, T - 5)

        tr = np.arange(train_T)
        te = np.arange(train_T, T)

        # ====================================================
        # Branch 2: centroid Bayesian online timing
        # ====================================================

        # Centroid extraction
        t0 = time.perf_counter()
        C = fill_nan_centroids(centroid_series(labels))
        centroid_time_s = time.perf_counter() - t0

        c0 = C[0]
        D = C - c0

        # Triplet construction
        t0 = time.perf_counter()
        Z, Vc, Ac = make_centroid_triplet(D, tr, KAPPA_ACC)
        centroid_triplet_time_s = time.perf_counter() - t0

        # Kalman prediction/update/surprise
        t0 = time.perf_counter()
        kalZ = kalman_observed_state(Z, tr)
        kalman_total_time_s = time.perf_counter() - t0

        # Gate decision
        t0 = time.perf_counter()
        surprise_train = kalZ["surprise"][tr]
        surprise_test = kalZ["surprise"][te]
        threshold = np.quantile(surprise_train[np.isfinite(surprise_train)], GATE_QUANTILE)
        rejected = surprise_test > threshold
        gate_time_s = time.perf_counter() - t0

        # Uncertainty covariance extraction
        t0 = time.perf_counter()
        pred_cov_test = kalZ["covs_pred"][te, :2, :2]
        pred_sigma_trace = np.array([np.trace(P) for P in pred_cov_test])
        uncertainty_extract_time_s = time.perf_counter() - t0

        # Prediction error, not an online clinical step, but timing negligible.
        pred_centroid = kalZ["means_pred"][:, :2] + c0
        err_test = np.linalg.norm(pred_centroid[te] - C[te], axis=1)

        # Estimate per-frame decomposition of Kalman total:
        # In this implementation prediction/update/surprise happen in one loop.
        kalman_surprise_ms_per_frame = 1000 * kalman_total_time_s / T

        # ====================================================
        # Branch 1: dense H&S online timing
        # ====================================================

        dense_available = False

        dense_crop_norm_time_s = np.nan
        hs_time_s = np.nan
        hs_ms_per_frame = np.nan
        dense_pod_projection_time_s = np.nan
        dense_triplet_projection_time_s = np.nan
        dense_residual_score_time_s = np.nan
        mask_warp_time_s = np.nan

        dense_pod_ms_per_frame = np.nan
        dense_triplet_pod_ms_per_frame = np.nan
        dense_residual_score_ms_per_frame = np.nan
        mask_warp_ms_per_frame = np.nan

        hs_cache_path = CACHEDIR / f"{case_id}_hs_dense_fields.npz"

        if USE_CACHED_HS_FIELDS and hs_cache_path.exists():
            dense_available = True

            dcache = np.load(hs_cache_path, allow_pickle=True)

            flows = dcache["flows"].astype(np.float32)
            frames_c = dcache["frames"].astype(np.float32)
            labels_c = dcache["labels"].astype(bool)

            Tc, Hc, Wc, _ = flows.shape
            P = Hc * Wc
            flow_dim = 2 * P

            # Use cached H&S timing from previous computation if available
            if "time_ms" in dcache.files:
                time_ms = dcache["time_ms"].astype(float)
                hs_ms_per_frame = float(np.nanmean(time_ms[1:]))
                hs_time_s = float(np.nansum(time_ms[1:]) / 1000)
            else:
                hs_time_s = np.nan
                hs_ms_per_frame = np.nan

            # Crop/normalization timing is measured fresh
            t0 = time.perf_counter()
            crop_box = crop_box_from_mask_union(labels, margin=60)
            frames_tmp, labels_tmp = crop_sequence(frames, labels, crop_box)
            frames_tmp = np.stack([normalize_image_np(f) for f in frames_tmp], axis=0)
            dense_crop_norm_time_s = time.perf_counter() - t0

            # Vectorize dense flow fields
            U = flows[..., 0].reshape(Tc, P)
            V = flows[..., 1].reshape(Tc, P)
            X = np.concatenate([U, V], axis=1).astype(np.float32)

            train_T_dense = max(8, int(round(TRAIN_FRAC * Tc)))
            train_T_dense = min(train_T_dense, Tc - 5)
            trd = np.arange(train_T_dense)
            ted = np.arange(train_T_dense, Tc)

            # Offline model fitting is not counted as online runtime.
            classical_model = fit_pod_runtime(X[trd])
            Zdense, lam1, lam2 = make_dense_triplet_state_runtime(X, trd, KAPPA_ACC)
            enriched_model = fit_pod_runtime(Zdense[trd])

            K_classical = min(5, classical_model["components"].shape[0])
            K_enriched = min(8, enriched_model["components"].shape[0])

            # Dense POD projection timing
            t0 = time.perf_counter()
            for t in ted:
                coeff = project_pod_runtime(X[t], classical_model, K_classical)
                xhat = reconstruct_pod_runtime(coeff, classical_model, K_classical)
            dense_pod_projection_time_s = time.perf_counter() - t0

            # Dense Triplet-POD projection timing
            t0 = time.perf_counter()
            for t in ted:
                coeff = project_pod_runtime(Zdense[t], enriched_model, K_enriched)
                zhat = reconstruct_pod_runtime(coeff, enriched_model, K_enriched)
            dense_triplet_projection_time_s = time.perf_counter() - t0

            # Dense residual score timing
            t0 = time.perf_counter()
            for t in ted:
                coeff = project_pod_runtime(Zdense[t], enriched_model, K_enriched)
                zhat = reconstruct_pod_runtime(coeff, enriched_model, K_enriched)
                score = np.sqrt(np.mean((zhat - Zdense[t]) ** 2))
            dense_residual_score_time_s = time.perf_counter() - t0

            # Mask warp timing
            ref_mask = labels_c[0]
            t0 = time.perf_counter()
            for t in ted:
                pred_mask = forward_warp_mask(ref_mask, flows[t])
            mask_warp_time_s = time.perf_counter() - t0

            dense_pod_ms_per_frame = 1000 * dense_pod_projection_time_s / len(ted)
            dense_triplet_pod_ms_per_frame = 1000 * dense_triplet_projection_time_s / len(ted)
            dense_residual_score_ms_per_frame = 1000 * dense_residual_score_time_s / len(ted)
            mask_warp_ms_per_frame = 1000 * mask_warp_time_s / len(ted)

        else:
            # If no cached H&S is available, we do not recompute by default.
            dense_available = False

        # ====================================================
        # Store case-level runtime
        # ====================================================

        branch2_online_ms_per_frame = (
            1000 * centroid_time_s / T
            + 1000 * centroid_triplet_time_s / T
            + 1000 * kalman_total_time_s / T
            + 1000 * uncertainty_extract_time_s / len(te)
            + 1000 * gate_time_s / len(te)
        )

        branch1_online_ms_per_frame_without_hs = np.nan
        branch1_online_ms_per_frame_with_hs = np.nan

        if dense_available:
            branch1_online_ms_per_frame_without_hs = (
                1000 * dense_crop_norm_time_s / T
                + dense_pod_ms_per_frame
                + dense_triplet_pod_ms_per_frame
                + dense_residual_score_ms_per_frame
                + mask_warp_ms_per_frame
            )

            branch1_online_ms_per_frame_with_hs = (
                branch1_online_ms_per_frame_without_hs
                + hs_ms_per_frame
            )

        runtime_rows.append({
            "case_id": case_id,
            "split": split,
            "T": int(T),
            "n_test": int(len(te)),

            # Branch availability
            "dense_branch_available": int(dense_available),

            # Data loading, not online
            "load_time_s": load_time_s,
            "load_ms_per_frame": 1000 * load_time_s / T,

            # Branch 2 online timings
            "centroid_extraction_ms_per_frame": 1000 * centroid_time_s / T,
            "centroid_triplet_update_ms_per_frame": 1000 * centroid_triplet_time_s / T,
            "kalman_prediction_update_surprise_ms_per_frame": kalman_surprise_ms_per_frame,
            "uncertainty_covariance_extract_ms_per_test_frame": 1000 * uncertainty_extract_time_s / len(te),
            "gate_decision_ms_per_test_frame": 1000 * gate_time_s / len(te),
            "branch2_centroid_bayesian_total_ms_per_frame": branch2_online_ms_per_frame,

            # Branch 2 performance context
            "rejection_rate": float(np.mean(rejected)),
            "mean_test_error_px": float(np.mean(err_test)),
            "p95_test_error_px": float(np.percentile(err_test, 95)),

            # Branch 1 timings
            "dense_crop_normalization_ms_per_frame": 1000 * dense_crop_norm_time_s / T if dense_available else np.nan,
            "gpu_hs_flow_ms_per_frame": hs_ms_per_frame,
            "dense_classical_pod_projection_ms_per_frame": dense_pod_ms_per_frame,
            "dense_triplet_pod_projection_ms_per_frame": dense_triplet_pod_ms_per_frame,
            "dense_triplet_residual_score_ms_per_frame": dense_residual_score_ms_per_frame,
            "mask_warp_ms_per_frame": mask_warp_ms_per_frame,
            "branch1_dense_total_without_hs_ms_per_frame": branch1_online_ms_per_frame_without_hs,
            "branch1_dense_total_with_hs_ms_per_frame": branch1_online_ms_per_frame_with_hs,
        })

    except Exception as e:
        runtime_rows.append({
            "case_id": case_id,
            "split": split,
            "error": str(e),
        })

global_runtime_s = time.perf_counter() - global_start

df_runtime = pd.DataFrame(runtime_rows)

case_runtime_path = RUNTIME_TABDIR / "paper_online_runtime_case_level.csv"
df_runtime.to_csv(case_runtime_path, index=False)

print("Saved case-level runtime:", case_runtime_path)
print("Total runtime measurement wall time:", round(global_runtime_s, 2), "s")

display(df_runtime.head(20))


# ============================================================
# Aggregate paper-ready runtime table
# ============================================================

runtime_columns = [
    "centroid_extraction_ms_per_frame",
    "centroid_triplet_update_ms_per_frame",
    "kalman_prediction_update_surprise_ms_per_frame",
    "uncertainty_covariance_extract_ms_per_test_frame",
    "gate_decision_ms_per_test_frame",
    "branch2_centroid_bayesian_total_ms_per_frame",

    "dense_crop_normalization_ms_per_frame",
    "gpu_hs_flow_ms_per_frame",
    "dense_classical_pod_projection_ms_per_frame",
    "dense_triplet_pod_projection_ms_per_frame",
    "dense_triplet_residual_score_ms_per_frame",
    "mask_warp_ms_per_frame",
    "branch1_dense_total_without_hs_ms_per_frame",
    "branch1_dense_total_with_hs_ms_per_frame",
]

summary_rows = []

for col in runtime_columns:
    if col not in df_runtime.columns:
        continue

    vals = df_runtime[col].values
    mean, lo, hi = boot_ci(vals)

    summary_rows.append({
        "online_process": col,
        "mean_ms_per_frame": mean,
        "ci95_low": lo,
        "ci95_high": hi,
        "n_cases": int(np.isfinite(vals).sum()),
    })

df_runtime_summary = pd.DataFrame(summary_rows)

summary_path = RUNTIME_TABDIR / "paper_online_runtime_summary.csv"
df_runtime_summary.to_csv(summary_path, index=False)

print("\n=== Paper-ready online runtime summary ===")
display(df_runtime_summary.round(4))

print("Saved runtime summary:", summary_path)


# ============================================================
# Optional: prettier labelled table
# ============================================================

label_map = {
    "centroid_extraction_ms_per_frame": "Target centroid extraction",
    "centroid_triplet_update_ms_per_frame": "Centroid triplet state update",
    "kalman_prediction_update_surprise_ms_per_frame": "Kalman prediction/update + surprise",
    "uncertainty_covariance_extract_ms_per_test_frame": "Uncertainty covariance extraction",
    "gate_decision_ms_per_test_frame": "Reliability gate decision",
    "branch2_centroid_bayesian_total_ms_per_frame": "Branch 2 total: centroid Bayesian",

    "dense_crop_normalization_ms_per_frame": "Dense crop + normalization",
    "gpu_hs_flow_ms_per_frame": "GPU Horn–Schunck flow",
    "dense_classical_pod_projection_ms_per_frame": "Dense classical POD projection",
    "dense_triplet_pod_projection_ms_per_frame": "Dense Triplet-POD projection",
    "dense_triplet_residual_score_ms_per_frame": "Dense Triplet residual score",
    "mask_warp_ms_per_frame": "Reference-mask propagation",
    "branch1_dense_total_without_hs_ms_per_frame": "Branch 1 total excluding H&S",
    "branch1_dense_total_with_hs_ms_per_frame": "Branch 1 total including H&S",
}

df_runtime_summary_pretty = df_runtime_summary.copy()
df_runtime_summary_pretty["online_process_label"] = df_runtime_summary_pretty["online_process"].map(label_map)

df_runtime_summary_pretty = df_runtime_summary_pretty[
    [
        "online_process_label",
        "mean_ms_per_frame",
        "ci95_low",
        "ci95_high",
        "n_cases",
    ]
]

pretty_path = RUNTIME_TABDIR / "paper_online_runtime_summary_pretty.csv"
df_runtime_summary_pretty.to_csv(pretty_path, index=False)

print("\n=== Pretty runtime table ===")
display(df_runtime_summary_pretty.round(4))

print("Saved pretty runtime table:", pretty_path)

In [ ]:
# ============================================================
# 7. Export all outputs
# ============================================================
zip_path = shutil.make_archive(str(OUTDIR), 'zip', OUTDIR)
print("Outputs folder:", OUTDIR)
print("Zip file:", zip_path)
print("Figures:")
for p in sorted(FIGDIR.glob("*.png")): print(" ",p)
print("Tables:")
for p in sorted(TABDIR.glob("*.csv")): print(" ",p)
